In [3062]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
import datetime as dt
import sqlite3
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats
import requests
from datetime import datetime, timedelta
import json


In [3063]:


class NBADataFetcher:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })

    def get_daily_lineups(self, date=None):
        """
        Fetch daily lineups for a specific date and return as DataFrame
        date: string in YYYYMMDD format, defaults to today
        """
        if date is None:
            date = datetime.now().strftime('%Y%m%d')
        url = f'https://stats.nba.com/js/data/leaders/00_daily_lineups_{date}.json'
        data = self._make_request(url)
        return self._parse_lineup_data(data) if data else None

    def _make_request(self, url):
        """Helper method to make requests with error handling"""
        try:
            response = self.session.get(url)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data from {url}: {str(e)}")
            return None

    def _parse_lineup_data(self, data):
        """Parse lineup data into DataFrame"""
        all_players = []
        
        for game in data['games']:
            game_id = game['gameId']
            game_status = game['gameStatusText']
            
            # Process home team players
            for player in game['homeTeam']['players']:
                player_data = {
                    'game_id': game_id,
                    'game_status': game_status,
                    'team': game['homeTeam']['teamAbbreviation'],
                    'is_home': True,
                    'player_id': player['personId'],
                    'player_name': player['playerName'],
                    'first_name': player['firstName'],
                    'last_name': player['lastName'],
                    'position': player['position'],
                    'lineup_status': player['lineupStatus']
                }
                all_players.append(player_data)
            
            # Process away team players
            for player in game['awayTeam']['players']:
                player_data = {
                    'game_id': game_id,
                    'game_status': game_status,
                    'team': game['awayTeam']['teamAbbreviation'],
                    'is_home': False,
                    'player_id': player['personId'],
                    'player_name': player['playerName'],
                    'first_name': player['firstName'],
                    'last_name': player['lastName'],
                    'position': player['position'],
                    'lineup_status': player['lineupStatus']
                }
                all_players.append(player_data)
        
        return pd.DataFrame(all_players)

def main():
    nba = NBADataFetcher()
    
    # Get lineups as DataFrame
    df_lineups = nba.get_daily_lineups()
    
    if df_lineups is not None:
        print("\nLineups DataFrame Shape:", df_lineups.shape)
        print("\nSample of Lineups DataFrame:")
        print(df_lineups.head())
        
        # Example: Get all players with "Out" status
        print("\nInjured/Out Players:")
        out_players = df_lineups[df_lineups['lineup_status'] == 'Out']
        print(out_players[['team', 'player_name', 'lineup_status']])
        
        # Example: Get starters for a specific team
        team = 'DET'  # Change this to any team abbreviation
        print(f"\nStarters for {team}:")
        starters = df_lineups[(df_lineups['team'] == team) & 
                             (df_lineups['lineup_status'] == 'Expected')]
        print(starters[['player_name', 'position', 'lineup_status']])

In [3064]:
# Connect to the database
conn = sqlite3.connect('nba_data.db') 

# Create a cursor object to execute SQL queries
cursor = conn.cursor()

In [3065]:
# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
tables

[('passing_df_data',),
 ('rebounding_df_data',),
 ('drives_df_data',),
 ('catchshoot_df_data',),
 ('pullup_df_data',),
 ('speeddistance_data',),
 ('posttouch_data',),
 ('player_type_defensive',),
 ('catchshoot_player_data',),
 ('pullup_player_data',),
 ('passing_data',),
 ('rebounding_player_data',),
 ('drives_player_data',),
 ('speeddistance_player_data',),
 ('posttouch_player_data',),
 ('painttouch_player_data',),
 ('shotclock_data',),
 ('closestdefender_data',),
 ('dribbles_shot_data',),
 ('touchtime_shot_data',),
 ('gamelogs',),
 ('player_points_scores',),
 ('player_3s_cluster',),
 ('player_3s_scores',),
 ('opponent_def_pts_rankings',),
 ('opponent_def_3pt_rankings',),
 ('opponent_def_type_scores',),
 ('closest_def_total',),
 ('playtype_off_reformat',),
 ('shotdetail_player_rolling',),
 ('drives_player_rolling',),
 ('catchshoot_player_rolling',),
 ('pullup_player_rolling',),
 ('rebounding_player_rolling',),
 ('speeddistance_player_rolling',),
 ('drives_team_def_rolling',),
 ('catch

In [3066]:
table_name = "closest_def_pts"
closest_def_pts = pd.read_sql(f"SELECT * FROM {table_name}", conn)
closest_def_pts['as_of'] = pd.to_datetime(closest_def_pts['as_of'])
closest_def_pts_c = closest_def_pts['as_of'].max()
closest_def_pts = closest_def_pts[closest_def_pts['as_of'] == closest_def_pts_c]

table_name = "closest_def_3pt"
closest_def_3pt = pd.read_sql(f"SELECT * FROM {table_name}", conn)
closest_def_3pt['as_of'] = pd.to_datetime(closest_def_3pt['as_of'])
closest_def_3pt_c = closest_def_3pt['as_of'].max()
closest_def_3pt = closest_def_3pt[closest_def_3pt['as_of'] == closest_def_3pt_c]

table_name = "closest_def_total"
closest_def_total = pd.read_sql(f"SELECT * FROM {table_name}", conn)
closest_def_total['as_of'] = pd.to_datetime(closest_def_total['as_of'])
closest_def_total_c = closest_def_total['as_of'].max()
closest_def_total = closest_def_total[closest_def_total['as_of'] == closest_def_total_c]

table_name = "opponent_def_pts_rankings"
opponent_def_pts_rankings = pd.read_sql(f"SELECT * FROM {table_name}", conn)
opponent_def_pts_rankings['as_of'] = pd.to_datetime(opponent_def_pts_rankings['as_of'])
opponent_def_pts_rankings_c = opponent_def_pts_rankings['as_of'].max()
opponent_def_pts_rankings = opponent_def_pts_rankings[opponent_def_pts_rankings['as_of'] == opponent_def_pts_rankings_c]

table_name = "opponent_def_3pt_rankings"
opponent_def_3pt_rankings = pd.read_sql(f"SELECT * FROM {table_name}", conn)
opponent_def_3pt_rankings['as_of'] = pd.to_datetime(opponent_def_3pt_rankings['as_of'])
opponent_def_3pt_rankings_c = opponent_def_3pt_rankings['as_of'].max()
opponent_def_3pt_rankings = opponent_def_3pt_rankings[opponent_def_3pt_rankings['as_of'] == opponent_def_3pt_rankings_c]

table_name = "opponent_def_Ast_rankings"
opponent_def_Ast_rankings = pd.read_sql(f"SELECT * FROM {table_name}", conn)
opponent_def_Ast_rankings['as_of'] = pd.to_datetime(opponent_def_Ast_rankings['as_of'])
opponent_def_Ast_rankings_c = opponent_def_Ast_rankings['as_of'].max()
opponent_def_Ast_rankings = opponent_def_Ast_rankings[opponent_def_Ast_rankings['as_of'] == opponent_def_Ast_rankings_c]

table_name = "opponent_def_reb_rankings"
opponent_def_reb_rankings = pd.read_sql(f"SELECT * FROM {table_name}", conn)
opponent_def_reb_rankings['as_of'] = pd.to_datetime(opponent_def_reb_rankings['as_of'])
opponent_def_reb_rankings_c = opponent_def_reb_rankings['as_of'].max()
opponent_def_reb_rankings = opponent_def_reb_rankings[opponent_def_reb_rankings['as_of'] == opponent_def_reb_rankings_c]

table_name = "opponent_def_type_scores_new"
opponent_def_type_scores = pd.read_sql(f"SELECT * FROM {table_name}", conn)
opponent_def_type_scores['as_of'] = pd.to_datetime(opponent_def_type_scores['as_of'])
opponent_def_type_scores_c = opponent_def_type_scores['as_of'].max()
opponent_def_type_scores = opponent_def_type_scores[opponent_def_type_scores['as_of'] == opponent_def_type_scores_c]

table_name = "def_3pt_scores"
def_3pt_scores = pd.read_sql(f"SELECT * FROM {table_name}", conn)
def_3pt_scores['as_of'] = pd.to_datetime(def_3pt_scores['as_of'])
def_3pt_scores_c = def_3pt_scores['as_of'].max()
def_3pt_scores = def_3pt_scores[def_3pt_scores['as_of'] == def_3pt_scores_c]

table_name = "def_ast_scores"
def_ast_scores = pd.read_sql(f"SELECT * FROM {table_name}", conn)
def_ast_scores['as_of'] = pd.to_datetime(def_ast_scores['as_of'])
def_ast_scores_c = def_ast_scores['as_of'].max()
def_ast_scores = def_ast_scores[def_ast_scores['as_of'] == def_ast_scores_c]

In [3067]:
#def_ast_scores

In [3068]:
from datetime import datetime
#today = '2025-01-31'
today = datetime.now().strftime('%Y-%m-%d')
today

'2025-03-24'

In [3069]:
from nba_api.stats.endpoints import leaguegamefinder


# Fetch all games for the 2023-24 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2023-24',league_id_nullable='00',season_type_nullable='Regular Season')
games = gamefinder.get_data_frames()[0]

# Display the first few rows
games_1 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
games_2 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
game_sched1 = games_1.merge(games_2, how='left',on=['GAME_ID','GAME_DATE','SEASON_ID'])
game_sched_23 = pd.DataFrame(game_sched1.loc[game_sched1['TEAM_ID_x']!=game_sched1['TEAM_ID_y']]).reset_index(drop=True)
game_sched_23.rename(columns={'TEAM_ID_x':'TEAM_ID','TEAM_ABBREVIATION_x':'TEAM_ABBREVIATION','TEAM_NAME_x':'TEAM_NAME','TEAM_ID_y':'OPPONENT_ID','TEAM_ABBREVIATION_y':'OPPONENT_ABBREVIATION','TEAM_NAME_y':'OPPONENT_NAME'},inplace=True)
game_sched_23['GAME_DATE'] = pd.to_datetime(game_sched_23['GAME_DATE'])


# Fetch all games for the 2023-24 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2024-25',league_id_nullable='00',season_type_nullable='Regular Season')
games = gamefinder.get_data_frames()[0]

# Display the first few rows
games_1 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
games_2 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
game_sched1 = games_1.merge(games_2, how='left',on=['GAME_ID','GAME_DATE','SEASON_ID'])
game_sched_24 = pd.DataFrame(game_sched1.loc[game_sched1['TEAM_ID_x']!=game_sched1['TEAM_ID_y']]).reset_index(drop=True)
game_sched_24.rename(columns={'TEAM_ID_x':'TEAM_ID','TEAM_ABBREVIATION_x':'TEAM_ABBREVIATION','TEAM_NAME_x':'TEAM_NAME','TEAM_ID_y':'OPPONENT_ID','TEAM_ABBREVIATION_y':'OPPONENT_ABBREVIATION','TEAM_NAME_y':'OPPONENT_NAME'},inplace=True)
game_sched_24['GAME_DATE'] = pd.to_datetime(game_sched_24['GAME_DATE'])

nba_sched = pd.concat([game_sched_23, game_sched_24])

In [3070]:
from nba_api.stats.endpoints import ScoreboardV2
from datetime import datetime
import pandas as pd


# Get the scoreboard for the specified date
scoreboard = ScoreboardV2(game_date=today)
    
# Get game headers which contain matchup info
games_df = scoreboard.game_header.get_data_frame()
    

In [3071]:
games_1 = games_df[['GAME_ID', 'HOME_TEAM_ID','VISITOR_TEAM_ID']].rename(columns={'HOME_TEAM_ID':'TEAM_ID','VISITOR_TEAM_ID':'OPPONENT_ID'})
games_2 = games_df[['GAME_ID', 'VISITOR_TEAM_ID', 'HOME_TEAM_ID']].rename(columns={'VISITOR_TEAM_ID':'TEAM_ID','HOME_TEAM_ID':'OPPONENT_ID'})
game_sched = pd.concat([games_1,games_2])


In [3072]:
#game_sched

In [3073]:
#game_sched

In [3074]:

def calculate_weighted_rolling_stats(
    gamelogs: pd.DataFrame,
    rolling_columns: list,
    window_size: int = 60,
    method: str = 'average'
) -> pd.DataFrame:
    """
    Calculate rolling statistics for basketball statistics.
    
    Parameters:
    -----------
    gamelogs : pd.DataFrame
        DataFrame containing game logs with required columns:
        - OPPONENT_ID
        - GAME_DATE
        - Statistical columns specified in rolling_columns
    rolling_columns : list
        List of statistical columns to calculate rolling stats for
    window_size : int
        Number of games to include in rolling window
    method : str
        'average' for rolling mean, 'sum' for rolling sum
    """
    if method not in ['average', 'sum']:
        raise ValueError("method must be either 'average' or 'sum'")
    
    # Validate required columns
    required_columns = {'PLAYER_ID', 'GAME_DATE'}
    missing_columns = required_columns - set(gamelogs.columns)
    if missing_columns:
        raise ValueError(f"DataFrame is missing required columns: {missing_columns}")
    
    # Validate statistical columns
    missing_stat_columns = set(rolling_columns) - set(gamelogs.columns)
    if missing_stat_columns:
        raise ValueError(f"DataFrame is missing specified statistical columns: {missing_stat_columns}")
    
    # Ensure gamelogs are sorted by date
    gamelogs['GAME_DATE'] = pd.to_datetime(gamelogs['GAME_DATE'])
    gamelogs_sorted = gamelogs.sort_values(by=['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)
    
    # Initialize list to store rolling calculations
    rolling_dfs = []
    
    # Calculate games count for each window
    games_in_window = (
        gamelogs_sorted
        .groupby('PLAYER_ID')['GAME_DATE']
        .rolling(window=window_size, min_periods=1)
        .count()
        .reset_index(level=0, drop=True)
        .rename(f'GAMES_IN_WINDOW_{window_size}G')
    )
    rolling_dfs.append(games_in_window)
    
    # Calculate rolling statistics for each statistical column
    for col in rolling_columns:
        try:
            if method == 'average':
                rolling_col = (
                    gamelogs_sorted
                    .groupby('PLAYER_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .mean()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Mavg'
            else:  # method == 'sum'
                rolling_col = (
                    gamelogs_sorted
                    .groupby('PLAYER_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .sum()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Sum'
            
            rolling_dfs.append(rolling_col.rename(f'{col}_{suffix}'))
            
        except Exception as e:
            print(f"Error processing column '{col}': {str(e)}")
            continue
    
    if len(rolling_dfs) <= 1:  # Only games count column
        raise ValueError("No statistical columns were successfully processed")
    
    # Combine all rolling statistics with original data
    rolling_df = pd.concat(rolling_dfs, axis=1)
    result_df = pd.concat(
        [gamelogs_sorted.reset_index(drop=True), rolling_df],
        axis=1
    )
    
    return result_df

In [3075]:
table_name = "player_scoring_clusters_new"
pts_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_3s_cluster_new"
threes_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_points_scores_new"
pts_scores_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_3s_scores"
threes_scores_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_passing_clusters"
pass_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_rebounding_clusters"
reb_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

pts_cluster_data['as_of'] = pd.to_datetime(pts_cluster_data['as_of'])
threes_cluster_data['as_of'] = pd.to_datetime(threes_cluster_data['as_of'])
pts_scores_data['as_of'] = pd.to_datetime(pts_scores_data['as_of'])
threes_scores_data['as_of'] = pd.to_datetime(threes_scores_data['as_of'])
pass_cluster_data['as_of'] = pd.to_datetime(pass_cluster_data['as_of'])
reb_cluster_data['as_of'] = pd.to_datetime(reb_cluster_data['as_of'])

max_date_pts_c = pts_cluster_data['as_of'].max()
max_date_3s_c = threes_cluster_data['as_of'].max()
max_date_pts_s = pts_scores_data['as_of'].max()
max_date_3s_s = threes_scores_data['as_of'].max()
max_date_asts_s = pass_cluster_data['as_of'].max()
max_date_reb_s = reb_cluster_data['as_of'].max()

pts_cluster_df = pts_cluster_data[pts_cluster_data['as_of'] == max_date_pts_c]
threes_cluster_df = threes_cluster_data[threes_cluster_data['as_of'] == max_date_3s_c]
pts_scores_df = pts_scores_data[pts_scores_data['as_of'] == max_date_pts_s]
threes_scores_df = threes_scores_data[threes_scores_data['as_of'] == max_date_3s_s]
ast_cluster_df = pass_cluster_data[pass_cluster_data['as_of'] == max_date_asts_s]
reb_cluster_df = reb_cluster_data[reb_cluster_data['as_of'] == max_date_reb_s]

In [3076]:
pts_cluster_add = pts_cluster_df[['PLAYER_ID','Cluster']].rename(columns={'Cluster':'Cluster_Pts'})
threes_cluster_add = threes_cluster_df[['PLAYER_ID','Cluster']].rename(columns={'Cluster':'Cluster_3pt'})

In [3077]:
table_name = "gamelogs"
gamelogs_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [3078]:
gamelogs_data['GAME_DATE'] = pd.to_datetime(gamelogs_data['GAME_DATE'])

In [3079]:
gamelogs_data = gamelogs_data.merge(pts_cluster_add, how='left')
gamelogs_data = gamelogs_data.merge(threes_cluster_add, how='left')
gamelogs_data = gamelogs_data.merge(ast_cluster_df[['PLAYER_ID','Passing_Cluster']], how='left')
gamelogs_data = gamelogs_data.merge(reb_cluster_df[['PLAYER_ID','Rebounding_Cluster']], how='left')

In [3080]:
gamelogs_data.drop_duplicates(inplace=True)

In [3081]:
 gamelogs_sorted = pd.DataFrame(gamelogs_data.sort_values(by=['PLAYER_ID','GAME_DATE'])).reset_index(drop=True)

In [3082]:
#gamelogs_sorted.drop_duplicates(subset=['PLAYER_NAME','TEAM_ID','GAME_ID'])

In [3083]:
thresholds_pts = [15, 20, 25, 30, 35, 40]
thresholds_3s = [3, 4, 5, 6, 7, 8]
thresholds_ast = [6, 8, 10, 12]
thresholds_reb= [6, 8, 10, 12, 14, 16]

for threshold in thresholds_pts:
    column_name = f"PTS_over_{threshold}"
    gamelogs_sorted[column_name] = gamelogs_sorted["PTS"].apply(lambda x: 1 if x >= threshold else 0)
for threshold in thresholds_3s:
    column_name = f"FG3M_over_{threshold}"
    gamelogs_sorted[column_name] = gamelogs_sorted["FG3M"].apply(lambda x: 1 if x >= threshold else 0)
for threshold in thresholds_ast:
    column_name = f"AST_over_{threshold}"
    gamelogs_sorted[column_name] = gamelogs_sorted["AST"].apply(lambda x: 1 if x >= threshold else 0)
for threshold in thresholds_reb:
    column_name = f"REB_over_{threshold}"
    gamelogs_sorted[column_name] = gamelogs_sorted["REB"].apply(lambda x: 1 if x >= threshold else 0)

In [3084]:
gamelogs_sorted_ = pd.DataFrame(gamelogs_sorted[['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP',
       'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM',
       'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK',
       'BLKA', 'PF', 'PFD', 'PTS', 
       'Cluster_Pts', 'Cluster_3pt','Passing_Cluster','Rebounding_Cluster', 'PTS_over_15', 'PTS_over_20',
       'PTS_over_25', 'PTS_over_30', 'PTS_over_35', 'PTS_over_40',
       'FG3M_over_3', 'FG3M_over_4', 'FG3M_over_5', 'FG3M_over_6',
       'FG3M_over_7', 'FG3M_over_8', 'AST_over_6', 'AST_over_8', 'AST_over_10',
       'AST_over_12','REB_over_6', 'REB_over_8', 'REB_over_10',
       'REB_over_12','REB_over_14','REB_over_16']])

In [3085]:
gamelogs_with_rolling = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_sorted_,
    rolling_columns=['MIN', 'PTS', 'REB', 'AST', 'FGA','FG3M','FG3A'])


gamelogs_with_rolling_Overs = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_with_rolling.drop(columns={'GAMES_IN_WINDOW_60G'}),
    rolling_columns=['PTS_over_15', 'PTS_over_20','PTS_over_25', 'PTS_over_30', 'PTS_over_35', 'PTS_over_40',
       'FG3M_over_3', 'FG3M_over_4', 'FG3M_over_5', 'FG3M_over_6', 'FG3M_over_7', 'FG3M_over_8', 'AST_over_6', 'AST_over_8', 'AST_over_10',
       'AST_over_12', 'REB_over_6', 'REB_over_8', 'REB_over_10',
       'REB_over_12','REB_over_14','REB_over_16'], 
    method='sum'
)

gamelogs_with_rolling10 = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_sorted_,
    window_size=10,
    rolling_columns=['MIN', 'PTS', 'REB', 'AST', 'FGA','FG3M','FG3A'])



In [3086]:
most_recent_mask = gamelogs_with_rolling10['GAME_DATE'] == gamelogs_with_rolling10.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
gamelogs_today_10 = gamelogs_with_rolling10[most_recent_mask]

gamelogs_today_10_ = gamelogs_today_10[['PLAYER_ID','PLAYER_NAME','TEAM_NAME','GAMES_IN_WINDOW_10G','MIN_10G_Mavg','PTS_10G_Mavg','REB_10G_Mavg','AST_10G_Mavg','FGA_10G_Mavg','FG3M_10G_Mavg','FG3A_10G_Mavg']]

In [3087]:
gamelogs_with_rolling_Overs['GAME_DATE'] = pd.to_datetime(gamelogs_with_rolling_Overs['GAME_DATE'])

In [3088]:
#gamelogs_with_rolling_Overs.loc[gamelogs_with_rolling_Overs['PLAYER_NAME'].str.contains('Karl-')]

In [3089]:
gamelogs_ALL = gamelogs_with_rolling_Overs.merge(nba_sched[['GAME_DATE','TEAM_ID','OPPONENT_ID','OPPONENT_NAME','OPPONENT_ABBREVIATION']], how='inner')

In [3090]:
#gamelogs_ALL

In [3091]:
gamelogs_ALL['id'] = gamelogs_ALL['PLAYER_ID'].astype(str) +'_'+ gamelogs_ALL['GAME_DATE'].astype(str) +"_"+ gamelogs_ALL['TEAM_ID'].astype(str)

In [3092]:
gamelogs_ALL_ = gamelogs_ALL[['PLAYER_ID', 'PLAYER_NAME',  'TEAM_ID',
       'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP',
        'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM',
       'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK',
       'BLKA', 'PF', 'PFD', 'PTS', 'Cluster_Pts', 'Cluster_3pt','Passing_Cluster','Rebounding_Cluster', 'PTS_over_15',
       'PTS_over_20', 'PTS_over_25', 'PTS_over_30', 'PTS_over_35',
       'PTS_over_40', 'FG3M_over_3', 'FG3M_over_4', 'FG3M_over_5',
       'FG3M_over_6', 'FG3M_over_7', 'FG3M_over_8', 'AST_over_6_60G_Sum',
       'AST_over_8_60G_Sum', 'AST_over_10_60G_Sum', 'AST_over_12_60G_Sum','REB_over_6_60G_Sum',
       'REB_over_8_60G_Sum', 'REB_over_10_60G_Sum', 'REB_over_12_60G_Sum',
       'REB_over_14_60G_Sum', 'REB_over_16_60G_Sum', 'MIN_60G_Mavg',
       'PTS_60G_Mavg', 'REB_60G_Mavg', 'AST_60G_Mavg', 'FGA_60G_Mavg',
       'FG3M_60G_Mavg', 'FG3A_60G_Mavg', 'GAMES_IN_WINDOW_60G',
       'OPPONENT_ID', 'OPPONENT_NAME', 'OPPONENT_ABBREVIATION', 'id']]

# MINUS 10 games ??

In [3093]:
most_recent_mask = gamelogs_with_rolling_Overs['GAME_DATE'] == gamelogs_with_rolling_Overs.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
gamelogs_today = gamelogs_with_rolling_Overs[most_recent_mask]

In [3094]:
gl_today = gamelogs_today[[ 'PLAYER_ID', 'PLAYER_NAME','TEAM_ID','TEAM_NAME','Cluster_Pts', 'Cluster_3pt','Passing_Cluster','Rebounding_Cluster', 'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg',
       'PTS_60G_Mavg', 'REB_60G_Mavg', 'AST_60G_Mavg', 'FGA_60G_Mavg',
       'FG3M_60G_Mavg', 'FG3A_60G_Mavg','PTS_over_15_60G_Sum', 'PTS_over_20_60G_Sum', 'PTS_over_25_60G_Sum',
       'PTS_over_30_60G_Sum', 'PTS_over_35_60G_Sum', 'PTS_over_40_60G_Sum','FG3M_over_3_60G_Sum', 'FG3M_over_4_60G_Sum', 'FG3M_over_5_60G_Sum',
       'FG3M_over_6_60G_Sum', 'FG3M_over_7_60G_Sum', 'FG3M_over_8_60G_Sum', 'AST_over_6_60G_Sum',
       'AST_over_8_60G_Sum', 'AST_over_10_60G_Sum', 'AST_over_12_60G_Sum','REB_over_6_60G_Sum',
       'REB_over_8_60G_Sum', 'REB_over_10_60G_Sum', 'REB_over_12_60G_Sum',
       'REB_over_14_60G_Sum', 'REB_over_16_60G_Sum']]

In [3095]:
gl_pts = gl_today.loc[gl_today['Cluster_Pts'].notna()]
gl_3pt = gl_today.loc[gl_today['Cluster_3pt'].notna()]
gl_Ast = gl_today.loc[gl_today['Passing_Cluster'].notna()]
gl_reb = gl_today.loc[gl_today['Rebounding_Cluster'].notna()]

In [3096]:
gl_pts_ = gl_pts.merge(gamelogs_today_10_, how='left')
gl_3pt_ = gl_3pt.merge(gamelogs_today_10_, how='left')
gl_Ast_ = gl_Ast.merge(gamelogs_today_10_, how='left')
gl_reb_ = gl_reb.merge(gamelogs_today_10_, how='left')

In [3097]:
gl_pts_today = gl_pts_[['PLAYER_ID','PLAYER_NAME','TEAM_ID','TEAM_NAME','Cluster_Pts','GAMES_IN_WINDOW_60G','MIN_60G_Mavg','PTS_60G_Mavg','FGA_60G_Mavg',
                        'GAMES_IN_WINDOW_10G','MIN_10G_Mavg','PTS_10G_Mavg','FGA_10G_Mavg','PTS_over_15_60G_Sum', 'PTS_over_20_60G_Sum', 'PTS_over_25_60G_Sum',
       'PTS_over_30_60G_Sum', 'PTS_over_35_60G_Sum', 'PTS_over_40_60G_Sum']].merge(game_sched, how='right')
gl_3pt_today = gl_3pt_[['PLAYER_ID','PLAYER_NAME','TEAM_ID','TEAM_NAME','Cluster_3pt','GAMES_IN_WINDOW_60G','MIN_60G_Mavg','FG3M_60G_Mavg','FG3A_60G_Mavg',
                        'GAMES_IN_WINDOW_10G','MIN_10G_Mavg','FG3M_10G_Mavg','FG3A_10G_Mavg','FG3M_over_3_60G_Sum', 'FG3M_over_4_60G_Sum', 'FG3M_over_5_60G_Sum',
       'FG3M_over_6_60G_Sum', 'FG3M_over_7_60G_Sum', 'FG3M_over_8_60G_Sum']].merge(game_sched, how='right')
gl_Ast_today = gl_Ast_[['PLAYER_ID','PLAYER_NAME','TEAM_ID','TEAM_NAME','Passing_Cluster','GAMES_IN_WINDOW_60G','MIN_60G_Mavg','AST_60G_Mavg',
                        'GAMES_IN_WINDOW_10G','MIN_10G_Mavg','AST_10G_Mavg','AST_over_6_60G_Sum',
       'AST_over_8_60G_Sum', 'AST_over_10_60G_Sum', 'AST_over_12_60G_Sum']].merge(game_sched, how='right')
gl_reb_today = gl_reb_[['PLAYER_ID','PLAYER_NAME','TEAM_ID','TEAM_NAME','Rebounding_Cluster','GAMES_IN_WINDOW_60G','MIN_60G_Mavg','REB_60G_Mavg',
                        'GAMES_IN_WINDOW_10G','MIN_10G_Mavg','REB_10G_Mavg','REB_over_6_60G_Sum',
       'REB_over_8_60G_Sum', 'REB_over_10_60G_Sum', 'REB_over_12_60G_Sum',
       'REB_over_14_60G_Sum', 'REB_over_16_60G_Sum']].merge(game_sched, how='right')

In [3098]:
pts_scores_ = pts_scores_df[['PLAYER_ID', 'PLAYER_NAME', 'penetrator_score',
       'pure_score', 'rim_runner_score', 'OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition',]]
three_scores_ = threes_scores_df[['PLAYER_ID', 'PLAYER_NAME', 'cse_score',
       'offscreen_cse_score', 'pnp_cse_score', 'pue_score', 'iso_pue_score',
       'corner_score',]]

In [3099]:
gl_pts_today_total = gl_pts_today.merge(pts_scores_,how='left')
gl_3s_today_total = gl_3pt_today.merge(three_scores_,how='left')

In [3100]:
scores_ = pts_scores_.merge(three_scores_, how='outer')

scores_add = scores_[['PLAYER_ID', 'PLAYER_NAME',  'penetrator_score',
       'pure_score', 'rim_runner_score', 'OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition', 'cse_score',
       'offscreen_cse_score', 'pnp_cse_score', 'pue_score', 'iso_pue_score',
       'corner_score']]

gamelogs_ALL_sc = gamelogs_ALL_.merge(scores_add, how ='left')

In [3101]:
opponent_def_3pt_= opponent_def_3pt_rankings[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_3pt', 'SEASON_YEAR', 'Games',
       'Avg_FG3M_Diff', 'Avg_FG3M_allowed', 'Avg_FG3A_Diff',]]

In [3102]:
closest_def_team3 = closest_def_total[['OPPONENT_ID', 'OPPONENT_NAME','defense3_norm_score', 'defense3_zscore']].rename(columns={'defense3_norm_score':'team3_norm_score','defense3_zscore':'team3_zscore'})

In [3103]:
closest_def_pts_ = closest_def_pts[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_Pts', 'FGA_Open_diff_60G_Mavg',
       'FGA_Wide_Open_diff_60G_Mavg', 'FGA_Tight_diff_60G_Mavg',
       'FGA_Very_Tight_diff_60G_Mavg', 'defense_norm_score', 'defense_zscore',
       'FG3A_Open_diff_60G_Mavg', 'FG3A_Wide_Open_diff_60G_Mavg',
       'FG3A_Tight_diff_60G_Mavg', 'FG3A_Very_Tight_diff_60G_Mavg',
       'defense3_norm_score', 'defense3_zscore',]]

closest_def_3pt_ = closest_def_3pt[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_3pt','open_defense_score','tight_defense_score','open_defense_rating','tight_defense_rating','defense3_score']].merge(closest_def_team3)

In [3104]:
#closest_def_3pt_['Cluster_rank_norm'] = closest_def_3pt_.groupby('Cluster_3pt')['defense3_norm_score'].rank(method='first', ascending=True)
closest_def_3pt_['Cluster_rank_zscore'] = closest_def_3pt_.groupby('Cluster_3pt')['defense3_score'].rank(method='first', ascending=True)


In [3105]:
gamelogs_ALL_sc_cls = gamelogs_ALL_sc.merge(closest_def_3pt_, how='left')

In [3106]:
opponent_def_type_scores.columns

Index(['OPPONENT_ID', 'OPPONENT_NAME', 'penetrator_score', 'pure_score',
       'rim_runner_score', 'OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition',
       'pullup_score', 'mid_score', 'fg3_score', 'driving_score', 'as_of',
       'id'],
      dtype='object')

In [3107]:


score_columns = ['penetrator_score', 'pure_score',
       'rim_runner_score', 'OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition']

# Keep ID and Name columns as is, add suffix to score columns
opponent_def_type_ = opponent_def_type_scores[['OPPONENT_ID', 'OPPONENT_NAME']].join(
    opponent_def_type_scores[score_columns].add_suffix('_def'))

In [3108]:
opponent_def_type_.columns

Index(['OPPONENT_ID', 'OPPONENT_NAME', 'penetrator_score_def',
       'pure_score_def', 'rim_runner_score_def', 'OVERALL_DEF_SCORE_Cut_def',
       'OVERALL_DEF_SCORE_Handoff_def', 'OVERALL_DEF_SCORE_Isolation_def',
       'OVERALL_DEF_SCORE_Misc_def', 'OVERALL_DEF_SCORE_OffRebound_def',
       'OVERALL_DEF_SCORE_OffScreen_def',
       'OVERALL_DEF_SCORE_PRBallHandler_def',
       'OVERALL_DEF_SCORE_PRRollMan_def', 'OVERALL_DEF_SCORE_Postup_def',
       'OVERALL_DEF_SCORE_Spotup_def', 'OVERALL_DEF_SCORE_Transition_def'],
      dtype='object')

In [3109]:
gamelogs_ALL_sc_cls_dsc = gamelogs_ALL_sc_cls.merge(opponent_def_type_, how='left')

In [3110]:
#opponent_def_pts_ranks

In [3111]:
opponent_def_pts_ranks = pd.DataFrame(opponent_def_pts_rankings[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_Pts', 'SEASON_YEAR', 'Games',
       'Avg_PTS_Diff', 'Avg_PTS_allowed', 'Avg_PTS_MIN_Diff']])
opponent_def_3pt_ranks = pd.DataFrame(opponent_def_3pt_rankings[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_3pt', 'SEASON_YEAR', 'Games',
       'Avg_FG3M_Diff', 'Avg_FG3M_allowed', 'Avg_FG3A_Diff',]])
opponent_def_Ast_ranks = pd.DataFrame(opponent_def_Ast_rankings[['OPPONENT_ID', 'OPPONENT_NAME', 'Passing_Cluster', 'SEASON_YEAR', 'Games',
       'Avg_Ast_allowed','Avg_Ast_Diff']])
opponent_def_reb_ranks = pd.DataFrame(opponent_def_reb_rankings[['OPPONENT_ID', 'OPPONENT_NAME', 'Rebounding_Cluster', 'SEASON_YEAR', 'Games',
       'Avg_Reb_allowed','Avg_Reb_Diff']])

In [3112]:
opponent_def_pts_ranks['SEASON_YEAR'] = opponent_def_pts_ranks['SEASON_YEAR'].replace('2023-24', 'Prior')
opponent_def_pts_ranks['SEASON_YEAR'] = opponent_def_pts_ranks['SEASON_YEAR'].replace('2024-25', 'Current')

opponent_def_3pt_ranks['SEASON_YEAR'] = opponent_def_3pt_ranks['SEASON_YEAR'].replace('2023-24', 'Prior')
opponent_def_3pt_ranks['SEASON_YEAR'] = opponent_def_3pt_ranks['SEASON_YEAR'].replace('2024-25', 'Current')

opponent_def_Ast_ranks['SEASON_YEAR'] = opponent_def_Ast_ranks['SEASON_YEAR'].replace('2023-24', 'Prior')
opponent_def_Ast_ranks['SEASON_YEAR'] = opponent_def_Ast_ranks['SEASON_YEAR'].replace('2024-25', 'Current')

opponent_def_reb_ranks['SEASON_YEAR'] = opponent_def_reb_ranks['SEASON_YEAR'].replace('2023-24', 'Prior')
opponent_def_reb_ranks['SEASON_YEAR'] = opponent_def_reb_ranks['SEASON_YEAR'].replace('2024-25', 'Current')

In [3113]:
opponent_def_pivot = opponent_def_pts_ranks.pivot(index=['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_Pts',], 
                    columns='SEASON_YEAR', 
                    values=['Games', 'Avg_PTS_Diff', 'Avg_PTS_allowed', 'Avg_PTS_MIN_Diff'])

opponent_def_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in opponent_def_pivot.columns]
opponent_def_pivot.reset_index(inplace=True)

opponent_def_3pt_pivot = opponent_def_3pt_ranks.pivot(index=['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_3pt',], 
                    columns='SEASON_YEAR', 
                    values=['Games','Avg_FG3M_Diff', 'Avg_FG3M_allowed', 'Avg_FG3A_Diff'])

opponent_def_3pt_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in opponent_def_3pt_pivot.columns]
opponent_def_3pt_pivot.reset_index(inplace=True)

opponent_def_Ast_pivot = opponent_def_Ast_ranks.pivot(index=['OPPONENT_ID', 'OPPONENT_NAME', 'Passing_Cluster',], 
                    columns='SEASON_YEAR', 
                    values=['Games', 'Avg_Ast_allowed', 'Avg_Ast_Diff'])

opponent_def_Ast_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in opponent_def_Ast_pivot.columns]
opponent_def_Ast_pivot.reset_index(inplace=True)

opponent_def_reb_pivot = opponent_def_reb_ranks.pivot(index=['OPPONENT_ID', 'OPPONENT_NAME', 'Rebounding_Cluster',], 
                    columns='SEASON_YEAR', 
                    values=['Games', 'Avg_Reb_allowed', 'Avg_Reb_Diff'])

opponent_def_reb_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in opponent_def_reb_pivot.columns]
opponent_def_reb_pivot.reset_index(inplace=True)

In [3114]:
opponent_def_ = pd.DataFrame(opponent_def_pivot[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_Pts', 'Games_Current',
       'Avg_PTS_Diff_Current','Avg_PTS_allowed_Current','Avg_PTS_MIN_Diff_Current','Games_Prior', 'Avg_PTS_Diff_Prior',
                    'Avg_PTS_allowed_Prior', 'Avg_PTS_MIN_Diff_Prior']])

opponent_def_3pt = pd.DataFrame(opponent_def_3pt_pivot[['OPPONENT_ID', 'OPPONENT_NAME', 'Cluster_3pt', 'Games_Current',
       'Avg_FG3M_Diff_Current','Avg_FG3M_allowed_Current','Avg_FG3A_Diff_Current','Games_Prior', 'Avg_FG3M_Diff_Prior',
                    'Avg_FG3M_allowed_Prior', 'Avg_FG3A_Diff_Prior']])
opponent_def_Ast = pd.DataFrame(opponent_def_Ast_pivot[['OPPONENT_ID', 'OPPONENT_NAME', 'Passing_Cluster', 'Games_Current',
       'Avg_Ast_Diff_Current','Avg_Ast_allowed_Current','Games_Prior', 'Avg_Ast_Diff_Prior',
                    'Avg_Ast_allowed_Prior']])

opponent_def_reb = pd.DataFrame(opponent_def_reb_pivot[['OPPONENT_ID', 'OPPONENT_NAME', 'Rebounding_Cluster', 'Games_Current',
       'Avg_Reb_Diff_Current','Avg_Reb_allowed_Current','Games_Prior', 'Avg_Reb_Diff_Prior',
                    'Avg_Reb_allowed_Prior']])

In [3115]:
gl_pts_scores = gl_pts_today_total.merge(opponent_def_type_)

In [3116]:
# First create the instance
nba = NBADataFetcher()

# Then get the lineups DataFrame
df_lineups = nba.get_daily_lineups()

In [3117]:
#df_lineups.loc[df_lineups['team']=='LAC']

In [3118]:
pts_scores_df = pd.DataFrame(gl_pts_scores[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'Cluster_Pts','OPPONENT_NAME',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg','GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'PTS_10G_Mavg', 'FGA_10G_Mavg','penetrator_score','penetrator_score_def', 'pure_score', 'pure_score_def',
       'rim_runner_score','rim_runner_score_def','OVERALL_SCORE_Cut','OVERALL_DEF_SCORE_Cut_def',
       'OVERALL_SCORE_Handoff','OVERALL_DEF_SCORE_Handoff_def','OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def',
       'OVERALL_SCORE_Misc', 'OVERALL_DEF_SCORE_Misc_def','OVERALL_SCORE_OffRebound','OVERALL_DEF_SCORE_OffRebound_def',
       'OVERALL_SCORE_OffScreen','OVERALL_DEF_SCORE_OffScreen_def',
       'OVERALL_SCORE_PRBallHandler','OVERALL_DEF_SCORE_PRBallHandler_def',
       'OVERALL_SCORE_PRRollMan','OVERALL_DEF_SCORE_PRRollMan_def', 'OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def',
       'OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def','OVERALL_SCORE_Transition','OVERALL_DEF_SCORE_Transition_def', 'PTS_over_15_60G_Sum', 'PTS_over_20_60G_Sum', 'PTS_over_25_60G_Sum',
       'PTS_over_30_60G_Sum', 'PTS_over_35_60G_Sum', 'PTS_over_40_60G_Sum']].drop_duplicates())

In [3119]:
pts_scores_df.loc[pts_scores_df['PLAYER_NAME']=='Jimmy Butler III']

,PLAYER_ID,PLAYER_NAME,TEAM_NAME,Cluster_Pts,OPPONENT_NAME,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,FGA_60G_Mavg,GAMES_IN_WINDOW_10G,MIN_10G_Mavg,PTS_10G_Mavg,FGA_10G_Mavg,penetrator_score,penetrator_score_def,pure_score,pure_score_def,rim_runner_score,rim_runner_score_def,OVERALL_SCORE_Cut,OVERALL_DEF_SCORE_Cut_def,OVERALL_SCORE_Handoff,OVERALL_DEF_SCORE_Handoff_def,OVERALL_SCORE_Isolation,OVERALL_DEF_SCORE_Isolation_def,OVERALL_SCORE_Misc,OVERALL_DEF_SCORE_Misc_def,OVERALL_SCORE_OffRebound,OVERALL_DEF_SCORE_OffRebound_def,OVERALL_SCORE_OffScreen,OVERALL_DEF_SCORE_OffScreen_def,OVERALL_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRBallHandler_def,OVERALL_SCORE_PRRollMan,OVERALL_DEF_SCORE_PRRollMan_def,OVERALL_SCORE_Postup,OVERALL_DEF_SCORE_Postup_def,OVERALL_SCORE_Spotup,OVERALL_DEF_SCORE_Spotup_def,OVERALL_SCORE_Transition,OVERALL_DEF_SCORE_Transition_def,PTS_over_15_60G_Sum,PTS_over_20_60G_Sum,PTS_over_25_60G_Sum,PTS_over_30_60G_Sum,PTS_over_35_60G_Sum,PTS_over_40_60G_Sum


In [3120]:
columns_to_round = [
   'MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg','MIN_10G_Mavg', 'PTS_10G_Mavg', 'FGA_10G_Mavg','penetrator_score','penetrator_score_def', 'pure_score', 'pure_score_def',
       'rim_runner_score','rim_runner_score_def','OVERALL_SCORE_Cut','OVERALL_DEF_SCORE_Cut_def',
       'OVERALL_SCORE_Handoff','OVERALL_DEF_SCORE_Handoff_def','OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def',
       'OVERALL_SCORE_Misc', 'OVERALL_DEF_SCORE_Misc_def','OVERALL_SCORE_OffRebound','OVERALL_DEF_SCORE_OffRebound_def',
       'OVERALL_SCORE_OffScreen','OVERALL_DEF_SCORE_OffScreen_def',
       'OVERALL_SCORE_PRBallHandler','OVERALL_DEF_SCORE_PRBallHandler_def',
       'OVERALL_SCORE_PRRollMan','OVERALL_DEF_SCORE_PRRollMan_def', 'OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def',
       'OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def','OVERALL_SCORE_Transition','OVERALL_DEF_SCORE_Transition_def',
    # Add your other columns here
]

pts_scores_df[columns_to_round] = pts_scores_df[columns_to_round].round(2)


In [3121]:
pts_scores_df['as_of'] = pd.to_datetime(today)
pts_scores_df['id'] = pts_scores_df['as_of'].astype(str)+"_"+pts_scores_df['PLAYER_ID'].astype(str)+pts_scores_df['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_pts_scores_today"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    pts_scores_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = pts_scores_df[~pts_scores_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_pts_scores_today' already exists. Checking for new records...
Inserted 125 new records into 'gl_pts_scores_today'.


In [3122]:
opponent_def_['Cluster_current_rank'] = opponent_def_.groupby('Cluster_Pts')['Avg_PTS_allowed_Current'].rank(method='first', ascending=True)
opponent_def_['Cluster_prior_rank'] = opponent_def_.groupby('Cluster_Pts')['Avg_PTS_allowed_Prior'].rank(method='first', ascending=True)

In [3123]:
gl_pts_cluster_def = gl_pts_today.merge(opponent_def_)

In [3124]:
columns_to_round = [
    'MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg', 'MIN_10G_Mavg', 'PTS_10G_Mavg', 'FGA_10G_Mavg','Avg_PTS_Diff_Current', 'Avg_PTS_allowed_Current',
       'Avg_PTS_MIN_Diff_Current','Avg_PTS_Diff_Prior',
       'Avg_PTS_allowed_Prior', 'Avg_PTS_MIN_Diff_Prior'
    # Add your other columns here
]

gl_pts_cluster_def[columns_to_round] = gl_pts_cluster_def[columns_to_round].round(2)


In [3125]:
gl_pts_cluster_def_ = pd.DataFrame(gl_pts_cluster_def[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT_NAME', 'Cluster_Pts',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg',
        'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'PTS_10G_Mavg', 'FGA_10G_Mavg',
       'PTS_over_15_60G_Sum', 'PTS_over_20_60G_Sum', 'PTS_over_25_60G_Sum',
       'PTS_over_30_60G_Sum', 'PTS_over_35_60G_Sum', 'PTS_over_40_60G_Sum', 'Games_Current',
       'Avg_PTS_Diff_Current', 'Avg_PTS_allowed_Current','Cluster_current_rank', 
       'Avg_PTS_MIN_Diff_Current', 'Games_Prior', 'Avg_PTS_Diff_Prior',
       'Avg_PTS_allowed_Prior', 'Cluster_prior_rank', 'Avg_PTS_MIN_Diff_Prior']])

In [3126]:
gl_pts_cluster_def_['as_of'] = pd.to_datetime(today)
gl_pts_cluster_def_['id'] = gl_pts_cluster_def_['as_of'].astype(str)+"_"+gl_pts_cluster_def_['PLAYER_ID'].astype(str)+gl_pts_cluster_def_['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_pts_cluster_def"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_pts_cluster_def_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_pts_cluster_def_[~gl_pts_cluster_def_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_pts_cluster_def' already exists. Checking for new records...
Inserted 125 new records into 'gl_pts_cluster_def'.


In [3127]:
gl_3pt_cluster_def = gl_3pt_today.merge(opponent_def_3pt)

In [3128]:
columns_to_round = [
    'MIN_60G_Mavg', 'FG3M_60G_Mavg','FG3A_60G_Mavg','MIN_10G_Mavg', 'FG3M_10G_Mavg','FG3A_10G_Mavg','Avg_FG3M_Diff_Current', 'Avg_FG3M_allowed_Current',
       'Avg_FG3A_Diff_Current', 'Avg_FG3M_Diff_Prior',
       'Avg_FG3M_allowed_Prior', 'Avg_FG3A_Diff_Prior'
    # Add your other columns here
]

gl_3pt_cluster_def[columns_to_round] = gl_3pt_cluster_def[columns_to_round].round(2)

In [3129]:
gl_3pt_cluster_def_ = pd.DataFrame(gl_3pt_cluster_def[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT_NAME' , 'Cluster_3pt',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg',
                                                       'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'FG3M_10G_Mavg', 'FG3A_10G_Mavg',
       'FG3M_over_3_60G_Sum', 'FG3M_over_4_60G_Sum', 'FG3M_over_5_60G_Sum',
       'FG3M_over_6_60G_Sum', 'FG3M_over_7_60G_Sum', 'FG3M_over_8_60G_Sum','Games_Current',
       'Avg_FG3M_Diff_Current', 'Avg_FG3M_allowed_Current',
       'Avg_FG3A_Diff_Current', 'Games_Prior', 'Avg_FG3M_Diff_Prior',
       'Avg_FG3M_allowed_Prior', 'Avg_FG3A_Diff_Prior']])

In [3130]:
gl_3pt_cluster_def_['as_of'] = pd.to_datetime(today)
gl_3pt_cluster_def_['id'] = gl_3pt_cluster_def_['as_of'].astype(str)+"_"+gl_3pt_cluster_def_['PLAYER_ID'].astype(str)+gl_3pt_cluster_def_['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_3pt_cluster_def"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_3pt_cluster_def_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_3pt_cluster_def_[~gl_3pt_cluster_def_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_3pt_cluster_def' already exists. Checking for new records...
Inserted 123 new records into 'gl_3pt_cluster_def'.


In [3131]:
gl_Ast_cluster_def = gl_Ast_today.merge(opponent_def_Ast)

In [3132]:
gl_Ast_cluster_def.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME', 'Passing_Cluster',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'AST_60G_Mavg',
       'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'AST_10G_Mavg',
       'AST_over_6_60G_Sum', 'AST_over_8_60G_Sum', 'AST_over_10_60G_Sum',
       'AST_over_12_60G_Sum', 'GAME_ID', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Games_Current', 'Avg_Ast_Diff_Current', 'Avg_Ast_allowed_Current',
       'Games_Prior', 'Avg_Ast_Diff_Prior', 'Avg_Ast_allowed_Prior'],
      dtype='object')

In [3133]:
columns_to_round = [
    'MIN_60G_Mavg', 'AST_60G_Mavg','MIN_10G_Mavg', 'AST_10G_Mavg',
       'AST_over_6_60G_Sum', 'AST_over_8_60G_Sum', 'AST_over_10_60G_Sum',
       'AST_over_12_60G_Sum', 'GAME_ID', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Games_Current', 'Avg_Ast_Diff_Current', 'Avg_Ast_allowed_Current',
       'Games_Prior', 'Avg_Ast_Diff_Prior', 'Avg_Ast_allowed_Prior'
    # Add your other columns here
]

gl_Ast_cluster_def[columns_to_round] = gl_Ast_cluster_def[columns_to_round].round(2)

In [3134]:
gl_Ast_cluster_def = gl_Ast_cluster_def.merge(ast_cluster_df[['PLAYER_ID','PASSES_MIN_60G_Sum','AST_PASS_60G_Sum','AST_PER_MIN','AST_TO_POTENTIAL',]], how='left')

In [3135]:
gl_Ast_cluster_def_ = pd.DataFrame(gl_Ast_cluster_def[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT_NAME' , 'Passing_Cluster',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'AST_60G_Mavg','GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'AST_10G_Mavg','PASSES_MIN_60G_Sum','AST_PASS_60G_Sum','AST_PER_MIN','AST_TO_POTENTIAL',
       'AST_over_6_60G_Sum', 'AST_over_8_60G_Sum', 'AST_over_10_60G_Sum',
       'AST_over_12_60G_Sum', 'Games_Current', 'Avg_Ast_Diff_Current', 'Avg_Ast_allowed_Current',
       'Games_Prior', 'Avg_Ast_Diff_Prior', 'Avg_Ast_allowed_Prior']])

In [3136]:
gl_Ast_cluster_def_ = gl_Ast_cluster_def_.merge(def_ast_scores[['OPPONENT_NAME', 'ast_shooter_score', 'ast_bigs_score',
       'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE', 'DRIVES_PASS_RATE',]], how='left')

In [3137]:
#gl_Ast_cluster_def_ = gl_Ast_cluster_def_.merge(ast_cluster_df[['PLAYER_ID','PASSES_MIN_60G_Sum','AST_PASS_60G_Sum','AST_PER_MIN','AST_TO_POTENTIAL','OVERALL_SCORE_PRBallHandler']], how='left')

In [3138]:
gl_Ast_cluster_def_['as_of'] = pd.to_datetime(today)
gl_Ast_cluster_def_['id'] = gl_Ast_cluster_def_['as_of'].astype(str)+"_"+gl_Ast_cluster_def_['PLAYER_ID'].astype(str)+gl_Ast_cluster_def_['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_ast_cluster_def"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_Ast_cluster_def_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_Ast_cluster_def_[~gl_Ast_cluster_def_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_ast_cluster_def' already exists. Checking for new records...
Inserted 78 new records into 'gl_ast_cluster_def'.


In [3139]:
gl_reb_cluster_def = gl_reb_today.merge(opponent_def_reb)

In [3140]:
gl_reb_cluster_def.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME',
       'Rebounding_Cluster', 'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg',
       'REB_60G_Mavg', 'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'REB_10G_Mavg',
       'REB_over_6_60G_Sum', 'REB_over_8_60G_Sum', 'REB_over_10_60G_Sum',
       'REB_over_12_60G_Sum', 'REB_over_14_60G_Sum', 'REB_over_16_60G_Sum',
       'GAME_ID', 'OPPONENT_ID', 'OPPONENT_NAME', 'Games_Current',
       'Avg_Reb_Diff_Current', 'Avg_Reb_allowed_Current', 'Games_Prior',
       'Avg_Reb_Diff_Prior', 'Avg_Reb_allowed_Prior'],
      dtype='object')

In [3141]:
columns_to_round = [
    'MIN_60G_Mavg', 'REB_60G_Mavg','MIN_10G_Mavg', 'REB_10G_Mavg',
       'REB_over_6_60G_Sum', 'REB_over_8_60G_Sum', 'REB_over_10_60G_Sum',
       'REB_over_12_60G_Sum', 'REB_over_14_60G_Sum', 'REB_over_16_60G_Sum', 'GAME_ID', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Games_Current','Avg_Reb_Diff_Current', 'Avg_Reb_allowed_Current', 'Games_Prior',
       'Avg_Reb_Diff_Prior', 'Avg_Reb_allowed_Prior'
    # Add your other columns here
]

gl_reb_cluster_def[columns_to_round] = gl_reb_cluster_def[columns_to_round].round(2)

In [3142]:
#gl_reb_cluster_def = gl_reb_cluster_def.merge(reb_cluster_df[['PLAYER_ID','PASSES_MIN_60G_Sum','AST_PASS_60G_Sum','AST_PER_MIN','AST_TO_POTENTIAL','OVERALL_SCORE_PRBallHandler']], how='left')

In [3143]:
gl_reb_cluster_def.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME',
       'Rebounding_Cluster', 'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg',
       'REB_60G_Mavg', 'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'REB_10G_Mavg',
       'REB_over_6_60G_Sum', 'REB_over_8_60G_Sum', 'REB_over_10_60G_Sum',
       'REB_over_12_60G_Sum', 'REB_over_14_60G_Sum', 'REB_over_16_60G_Sum',
       'GAME_ID', 'OPPONENT_ID', 'OPPONENT_NAME', 'Games_Current',
       'Avg_Reb_Diff_Current', 'Avg_Reb_allowed_Current', 'Games_Prior',
       'Avg_Reb_Diff_Prior', 'Avg_Reb_allowed_Prior'],
      dtype='object')

In [3144]:
gl_reb_cluster_def_ = pd.DataFrame(gl_reb_cluster_def[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT_NAME' , 'Rebounding_Cluster', 'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg',
       'REB_60G_Mavg', 'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg',
       'REB_10G_Mavg','REB_over_6_60G_Sum', 'REB_over_8_60G_Sum',
       'REB_over_10_60G_Sum', 'REB_over_12_60G_Sum', 'REB_over_14_60G_Sum',
       'REB_over_16_60G_Sum', 
       'Games_Current', 'Avg_Reb_Diff_Current', 'Avg_Reb_allowed_Current',
       'Games_Prior', 'Avg_Reb_Diff_Prior', 'Avg_Reb_allowed_Prior']])

In [3145]:
gl_reb_cluster_def_['as_of'] = pd.to_datetime(today)
gl_reb_cluster_def_['id'] = (
    gl_reb_cluster_def_['as_of'].fillna('').astype(str) + "_" + 
    gl_reb_cluster_def_['PLAYER_ID'].fillna('').astype(str) + "_" + 
    gl_reb_cluster_def_['OPPONENT_NAME'].fillna('').astype(str)
)
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_reb_cluster_def"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_reb_cluster_def_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_reb_cluster_def_[~gl_reb_cluster_def_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_reb_cluster_def' already exists. Checking for new records...
Inserted 126 new records into 'gl_reb_cluster_def'.


In [3146]:

 gl_3pt_closest_def = gl_3pt_today.merge(closest_def_3pt_)

In [3147]:
columns_to_round = [
    'MIN_60G_Mavg', 'FG3M_60G_Mavg','FG3A_60G_Mavg','MIN_10G_Mavg', 'FG3M_10G_Mavg','FG3A_10G_Mavg',
    # Add your other columns here
]

gl_3pt_closest_def[columns_to_round] = gl_3pt_closest_def[columns_to_round].round(2)

In [3148]:
gl_3pt_closest_def_ = pd.DataFrame(gl_3pt_closest_def[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT_NAME', 'Cluster_3pt',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg','GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'FG3M_10G_Mavg', 'FG3A_10G_Mavg', 'open_defense_score','tight_defense_score','open_defense_rating','tight_defense_rating','defense3_score','team3_norm_score', 'team3_zscore',
       'Cluster_rank_zscore']])

In [3149]:
gl_Ast_cluster_def_ = gl_Ast_cluster_def_.merge(def_ast_scores[['OPPONENT_NAME', 'ast_shooter_score', 'ast_bigs_score',
       'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE', 'DRIVES_PASS_RATE',]], how='left')

In [3150]:
gl_Ast_cluster_def_['as_of'] = pd.to_datetime(today)
gl_Ast_cluster_def_['id'] = gl_Ast_cluster_def_['as_of'].astype(str)+"_"+gl_Ast_cluster_def_['PLAYER_ID'].astype(str)+gl_Ast_cluster_def_['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_ast_cluster_def"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_Ast_cluster_def_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_Ast_cluster_def_[~gl_Ast_cluster_def_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_ast_cluster_def' already exists. Checking for new records...
No new records to insert.


In [3151]:
gl_3pt_closest_def_['as_of'] = pd.to_datetime(today)
gl_3pt_closest_def_['id'] = gl_3pt_closest_def_['as_of'].astype(str)+"_"+gl_3pt_closest_def_['PLAYER_ID'].astype(str)+gl_3pt_closest_def_['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_3pt_closest_def"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_3pt_closest_def_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_3pt_closest_def_[~gl_3pt_closest_def_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_3pt_closest_def' already exists. Checking for new records...
Inserted 123 new records into 'gl_3pt_closest_def'.


In [3152]:

score_columns = [ 'cse_score', 'offscreen_cse_score',
       'pnp_cse_score', 'pue_score', 'iso_pue_score', 'corner_score']

# Keep ID and Name columns as is, add suffix to score columns
def_3pt_scores_ = def_3pt_scores[['OPPONENT_ID', 'OPPONENT_NAME']].join(
    def_3pt_scores[score_columns].add_suffix('_def'))

In [3153]:
gamelogs_ALL_sc_cls_dsc_3sc = gamelogs_ALL_sc_cls_dsc.merge(def_3pt_scores_)

In [3154]:
gl_3s_today_scores = gl_3s_today_total.merge(def_3pt_scores_, how='left')

In [3155]:
gl_3s_today_scores_ = pd.DataFrame(gl_3s_today_scores[['PLAYER_ID', 'PLAYER_NAME','TEAM_NAME','OPPONENT_NAME', 'Cluster_3pt',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg','GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'FG3M_10G_Mavg', 'FG3A_10G_Mavg',
       'FG3M_over_3_60G_Sum', 'FG3M_over_4_60G_Sum', 'FG3M_over_5_60G_Sum',
       'FG3M_over_6_60G_Sum', 'FG3M_over_7_60G_Sum', 'FG3M_over_8_60G_Sum', 'cse_score', 'cse_score_def', 'offscreen_cse_score','offscreen_cse_score_def',
       'pnp_cse_score','pnp_cse_score_def', 'pue_score','pue_score_def',  'iso_pue_score','iso_pue_score_def', 'corner_score','corner_score_def']])

In [3156]:
gl_3s_today_scores_['as_of'] = pd.to_datetime(today)
gl_3s_today_scores_['id'] = gl_3s_today_scores_['as_of'].astype(str)+"_"+gl_3s_today_scores_['PLAYER_ID'].astype(str)+gl_3s_today_scores_['OPPONENT_NAME'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gl_3s_today_scores"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gl_3s_today_scores_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gl_3s_today_scores_[~gl_3s_today_scores_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gl_3s_today_scores' already exists. Checking for new records...
Inserted 123 new records into 'gl_3s_today_scores'.


In [3157]:
gamelogs_ALL_sc_cls_dsc_3sc.drop(columns='id', inplace=True)

In [3158]:
gamelogs_ALL_sc_cls_dsc_3sc1 = gamelogs_ALL_sc_cls_dsc_3sc.merge(opponent_def_, how='left')


In [3159]:
opponent_def_reb['Reb_current_rank'] = opponent_def_reb.groupby('Rebounding_Cluster')['Avg_Reb_Diff_Current'].rank(method='first', ascending=True)
opponent_def_Ast['Ast_current_rank'] = opponent_def_Ast.groupby('Passing_Cluster')['Avg_Ast_Diff_Current'].rank(method='first', ascending=True)
opponent_def_reb['Reb_prior_rank'] = opponent_def_reb.groupby('Rebounding_Cluster')['Avg_Reb_Diff_Prior'].rank(method='first', ascending=True)
opponent_def_Ast['Ast_prior_rank'] = opponent_def_Ast.groupby('Passing_Cluster')['Avg_Ast_Diff_Prior'].rank(method='first', ascending=True)

In [3160]:
opponent_def_reb.rename(columns={'Games_Current':'Games_Current_reb','Games_Prior':'Games_Prior_reb'}, inplace=True)
opponent_def_Ast.rename(columns={'Games_Current':'Games_Current_ast','Games_Prior':'Games_Prior_ast'}, inplace=True)


In [3161]:
gamelogs_ALL_sc_cls_dsc_3sc_reb = gamelogs_ALL_sc_cls_dsc_3sc1.merge(opponent_def_reb, how='left')
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast = gamelogs_ALL_sc_cls_dsc_3sc_reb.merge(opponent_def_Ast, how='left')
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1 = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast.merge(def_ast_scores[['OPPONENT_ID', 'OPPONENT_NAME', 'ast_shooter_score', 'ast_bigs_score',
       'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE', 'DRIVES_PASS_RATE']], how='left')

gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1 = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.merge(gamelogs_today_10_, how='left')

In [3162]:
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['id'] = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE'].astype(str)+"_"+gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_ID'].astype(str)+gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['OPPONENT_ID'].astype(str)

gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.sort_values(by='GAME_DATE', inplace=True, ascending=False)

In [3163]:
gl_3pt_cluster_def_.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT_NAME', 'Cluster_3pt',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg',
       'GAMES_IN_WINDOW_10G', 'MIN_10G_Mavg', 'FG3M_10G_Mavg', 'FG3A_10G_Mavg',
       'FG3M_over_3_60G_Sum', 'FG3M_over_4_60G_Sum', 'FG3M_over_5_60G_Sum',
       'FG3M_over_6_60G_Sum', 'FG3M_over_7_60G_Sum', 'FG3M_over_8_60G_Sum',
       'Games_Current', 'Avg_FG3M_Diff_Current', 'Avg_FG3M_allowed_Current',
       'Avg_FG3A_Diff_Current', 'Games_Prior', 'Avg_FG3M_Diff_Prior',
       'Avg_FG3M_allowed_Prior', 'Avg_FG3A_Diff_Prior', 'as_of', 'id'],
      dtype='object')

In [3164]:
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1 = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.merge(opponent_def_3pt[[ 'Cluster_3pt','OPPONENT_NAME','Avg_FG3M_Diff_Current', 'Avg_FG3M_allowed_Current',
       'Avg_FG3A_Diff_Current','Avg_FG3M_Diff_Prior',
       'Avg_FG3M_allowed_Prior', 'Avg_FG3A_Diff_Prior']], how='left')

In [3165]:
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.drop_duplicates(inplace=True)

In [3166]:
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['cse_score_def'] > 80)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_NAME'] =='Pascal Siakam')].head(10)

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,Cluster_Pts,Cluster_3pt,Passing_Cluster,Rebounding_Cluster,PTS_over_15,PTS_over_20,PTS_over_25,PTS_over_30,PTS_over_35,PTS_over_40,FG3M_over_3,FG3M_over_4,FG3M_over_5,FG3M_over_6,FG3M_over_7,FG3M_over_8,AST_over_6_60G_Sum,AST_over_8_60G_Sum,AST_over_10_60G_Sum,AST_over_12_60G_Sum,REB_over_6_60G_Sum,REB_over_8_60G_Sum,REB_over_10_60G_Sum,REB_over_12_60G_Sum,REB_over_14_60G_Sum,REB_over_16_60G_Sum,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,FGA_60G_Mavg,FG3M_60G_Mavg,FG3A_60G_Mavg,GAMES_IN_WINDOW_60G,OPPONENT_ID,OPPONENT_NAME,OPPONENT_ABBREVIATION,penetrator_score,pure_score,rim_runner_score,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,cse_score,offscreen_cse_score,pnp_cse_score,pue_score,iso_pue_score,corner_score,open_defense_score,tight_defense_score,open_defense_rating,tight_defense_rating,defense3_score,team3_norm_score,team3_zscore,Cluster_rank_zscore,penetrator_score_def,pure_score_def,rim_runner_score_def,OVERALL_DEF_SCORE_Cut_def,OVERALL_DEF_SCORE_Handoff_def,OVERALL_DEF_SCORE_Isolation_def,OVERALL_DEF_SCORE_Misc_def,OVERALL_DEF_SCORE_OffRebound_def,OVERALL_DEF_SCORE_OffScreen_def,OVERALL_DEF_SCORE_PRBallHandler_def,OVERALL_DEF_SCORE_PRRollMan_def,OVERALL_DEF_SCORE_Postup_def,OVERALL_DEF_SCORE_Spotup_def,OVERALL_DEF_SCORE_Transition_def,cse_score_def,offscreen_cse_score_def,pnp_cse_score_def,pue_score_def,iso_pue_score_def,corner_score_def,Games_Current,Avg_PTS_Diff_Current,Avg_PTS_allowed_Current,Avg_PTS_MIN_Diff_Current,Games_Prior,Avg_PTS_Diff_Prior,Avg_PTS_allowed_Prior,Avg_PTS_MIN_Diff_Prior,Cluster_current_rank,Cluster_prior_rank,Games_Current_reb,Avg_Reb_Diff_Current,Avg_Reb_allowed_Current,Games_Prior_reb,Avg_Reb_Diff_Prior,Avg_Reb_allowed_Prior,Reb_current_rank,Reb_prior_rank,Games_Current_ast,Avg_Ast_Diff_Current,Avg_Ast_allowed_Current,Games_Prior_ast,Avg_Ast_Diff_Prior,Avg_Ast_allowed_Prior,Ast_current_rank,Ast_prior_rank,ast_shooter_score,ast_bigs_score,DRIVES/MIN_60Day,DRIVES_AST_PASS_RATE,DRIVES_PASS_RATE,GAMES_IN_WINDOW_10G,MIN_10G_Mavg,PTS_10G_Mavg,REB_10G_Mavg,AST_10G_Mavg,FGA_10G_Mavg,FG3M_10G_Mavg,FG3A_10G_Mavg,id,Avg_FG3M_Diff_Current,Avg_FG3M_allowed_Current,Avg_FG3A_Diff_Current,Avg_FG3M_Diff_Prior,Avg_FG3M_allowed_Prior,Avg_FG3A_Diff_Prior
363,1627783,Pascal Siakam,1610612754,IND,Indiana Pacers,0022400942,2025-03-11,IND vs. MIL,37.393333,10,15,0.667,3,7,0.429,2,3,0.667,2,10,12,5,3,1,0,1,3,6,25,5.0,4.0,2.0,1.0,1,1,1,0,0,0,1,0,0,0,0,0,7.0,0.0,0.0,0.0,45.0,26.0,11.0,3.0,1.0,1.0,32.947278,20.983333,7.283333,3.183333,15.383333,1.800000,4.366667,60.0,1610612749,Milwaukee Bucks,MIL,94.369842,117.783251,132.120276,45.7,41.9,50.5,56.7,48.1,17.7,25.6,49.9,68.6,53.8,63.6,110.771976,95.342334,142.329678,23.319181,68.651174,100.434265,0.664184,0.144737,Poor,Poor,0.404460,5.79,6.33,29.0,109.020363,90.583082,76.71,26.6,32.7,36.4,59.3,35.1,56.4,65.6,25.8,18.6,47.1,56.3,112.239815,108.976850,91.434767,91.213131,89.818055,130.805630,36.0,0.365833,21.611111,0.017838,55.0,2.070364,22.472727,0.045893,9.0,18.0,66.0,-0.120455,8.909091,84.0,0.140952,9.095238,10.0,17.0,27.0,-0.404074,3.185185,41.0,0.426098,4.414634,11.0,26.0,108.139388,74.776568,9.23,0.28,0.41,10.0,31.555667,21.6,7.2,2.9,16.2,1.9,5.1,2025-03-11_16277831610612749,0.125412,1.529412,0.610353,-0.092083,1.302083,0.031042
958,1627783,Pascal Siakam,1610612754,IND,Indiana Pacers,0022400914,2025-03-08,IND @ ATL,33.633333,8,19,0.421,1,5,0.200,6,6,1.000,2,5,7,3,1,2,1,1,3,6,23,5.0,4.0,2.0,1.0,1,1,0,0,0,0,0,0,0,0,0,0,8.0,0.0,0.0,0.0,44.0,25.0,10.0,2.0,1.0,1.0,32.976000,20.833333,7.233333,3.166667,15.300000,1.766667,4.316667,60.0,1610612737,Atlanta Hawks,ATL,94.369842,117.783251

In [3167]:

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "gamelogs_all_detail"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1[~gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'gamelogs_all_detail' already exists. Checking for new records...
No new records to insert.


In [3168]:
import gspread
from gspread_dataframe import set_with_dataframe


In [3169]:
gc = gspread.service_account(filename='friend.json')

In [3170]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('gl_3s_today_scores')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, gl_3s_today_scores_)


In [3171]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('gl_3pt_closest_def_')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, gl_3pt_closest_def_)

In [3172]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('gl_3pt_cluster_def_')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, gl_3pt_cluster_def_)

In [3173]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('gl_pts_cluster_def_')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, gl_pts_cluster_def_)

In [3174]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('pts_scores_df')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, pts_scores_df)

In [3175]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('ast_cluster_df')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, gl_Ast_cluster_def_)

In [3176]:
# Open the Google Sheet by name
spreadsheet = gc.open('NBA 2024')  # Open the Google Sheet

# Select the specific worksheet/sheet within the spreadsheet
# Replace 'Sheet1' with the actual sheet name if it is different
sheet = spreadsheet.worksheet('reb_cluster_df')  # Or use `sheet1` if it's the first sheet

# Send the DataFrame to Google Sheets
set_with_dataframe(sheet, gl_reb_cluster_def_)

In [3177]:
#pts_scores_df.to_csv('pts_scores_df.csv')

In [3178]:
def filter_dataframe(df, filters=None, or_filters=None):
    """
    Filter DataFrame based on optional filter conditions with AND and OR support.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame to filter
    filters : dict, optional
        Dictionary of AND filter conditions
    or_filters : list of dict, optional
        List of dictionaries where each dict contains filter conditions to be OR'ed
        Each dict follows same format as filters parameter
    
    Returns:
    --------
    pandas.DataFrame
        Filtered DataFrame
    """
    if filters is None and or_filters is None:
        return df
    
    filtered_df = df.copy()
    
    # Apply AND filters
    if filters is not None:
        for column, conditions in filters.items():
            if column not in df.columns:
                continue
                
            operator = conditions.get('operator')
            value = conditions.get('value')
            
            if operator == '>':
                filtered_df = filtered_df[filtered_df[column] > value]
            elif operator == '<':
                filtered_df = filtered_df[filtered_df[column] < value]
            elif operator == '==':
                filtered_df = filtered_df[filtered_df[column] == value]
            elif operator == '>=':
                filtered_df = filtered_df[filtered_df[column] >= value]
            elif operator == '<=':
                filtered_df = filtered_df[filtered_df[column] <= value]
            elif operator == 'in':
                if isinstance(value, (list, tuple)):
                    filtered_df = filtered_df[filtered_df[column].isin(value)]
            elif operator == 'between':
                if isinstance(value, (list, tuple)) and len(value) == 2:
                    filtered_df = filtered_df[
                        (filtered_df[column] >= value[0]) & 
                        (filtered_df[column] <= value[1])
                    ]
    
    # Apply OR filters
    if or_filters is not None:
        or_conditions = pd.Series(False, index=filtered_df.index)
        
        for or_filter in or_filters:
            temp_condition = pd.Series(True, index=filtered_df.index)
            
            for column, conditions in or_filter.items():
                if column not in df.columns:
                    continue
                    
                operator = conditions.get('operator')
                value = conditions.get('value')
                
                if operator == '>':
                    temp_condition &= (filtered_df[column] > value)
                elif operator == '<':
                    temp_condition &= (filtered_df[column] < value)
                elif operator == '==':
                    temp_condition &= (filtered_df[column] == value)
                elif operator == '>=':
                    temp_condition &= (filtered_df[column] >= value)
                elif operator == '<=':
                    temp_condition &= (filtered_df[column] <= value)
                elif operator == 'in':
                    if isinstance(value, (list, tuple)):
                        temp_condition &= (filtered_df[column].isin(value))
                elif operator == 'between':
                    if isinstance(value, (list, tuple)) and len(value) == 2:
                        temp_condition &= (
                            (filtered_df[column] >= value[0]) & 
                            (filtered_df[column] <= value[1])
                        )
            
            or_conditions |= temp_condition
        
        filtered_df = filtered_df[or_conditions]
    
    return filtered_df

# Example usage
# AND filters
filters = {
    'MIN': {'operator': '>=', 'value': 20}
}

# OR filters
or_filters = [
    {'cse_score_def': {'operator': '>', 'value': 80}},
    {'PLAYER_NAME': {'operator': '==', 'value': 'Pascal Siakam'}}
]

# Apply both AND and OR filters
filtered_df = filter_dataframe(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1, 
                             filters=filters, 
                             or_filters=or_filters)

In [3179]:
this_year = pd.to_datetime("10-20-2024")

In [3180]:
player= "Mike Conley"

In [2944]:

# Example usage
filters = {
    'GAME_DATE': {'operator': '>', 'value': this_year},
#    'OPPONENT_NAME': {'operator': '==', 'value': 'Miami Heat'},
#    Threes
#    'cse_score_def': {'operator': '>', 'value': 100},
#    'offscreen_cse_score_def': {'operator': '>', 'value': 100},
#    'pnp_cse_score_def': {'operator': '>', 'value': 100},
#    'pue_score_def': {'operator': '>', 'value': 100},
#    'iso_pue_score_def': {'operator': '>', 'value': 110},
#    'corner_score_def': {'operator': '>', 'value': 100},
    #'open_defense_rating': {'operator': '==', 'value': "Below Average"},
#    Points
#    'penetrator_score_def': {'operator': '>', 'value': 100},
#    'pure_score_def': {'operator': '>', 'value': 100},
#    'rim_runner_score_def': {'operator': '>', 'value': 100},
#    'Cluster_current_rank': {'operator': '>', 'value': 100},
#    'Cluster_prior_rank': {'operator': '>', 'value': 100},
#    'OVERALL_DEF_SCORE_PRBallHandler_def': {'operator': '>', 'value': 100},
#    'OVERALL_DEF_SCORE_Isolation_def': {'operator': '>', 'value': 100},
#    'OVERALL_DEF_SCORE_PRRollMan_def': {'operator': '>', 'value': 100},
#    'OVERALL_DEF_SCORE_Transition_def': {'operator': '>', 'value': 100},
#    Ast
#    'Avg_Ast_Diff_Current': {'operator': '>', 'value': 0},
#    'Avg_Ast_Diff_Prior': {'operator': '>', 'value': 0},
#    'ast_shooter_score': {'operator': '>', 'value': 100},
#    'ast_bigs_score': {'operator': '>', 'value': 100},
#    Reb
 #   'Avg_Reb_Diff_Current': {'operator': '>', 'value': 0},
 #   'Avg_Reb_Diff_Prior': {'operator': '>', 'value': 0},
# Player    
    'PLAYER_NAME': {'operator': '==', 'value': player}
}

# Example usage
# AND filters


# OR filters
or_filters = [
    {'open_defense_rating': {'operator': '==', 'value': "Below Average"}},
    {'open_defense_rating': {'operator': '==', 'value': "Poor"}}
]


'''
filters = {
    'PTS': {'operator': 'between', 'value': [20, 30]},
    'PLAYER_NAME': {'operator': 'in', 'value': ['Pascal Siakam', 'Joel Embiid']}
}'''
# Filter with conditions
filtered_df = filter_dataframe(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1, 
                             filters=filters, 
                             or_filters=or_filters)

# Without filters (returns all data)
#all_data = filter_dataframe(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1)

# 3pt

In [2352]:
player= "Tyrese Maxey"

In [2353]:


filters = {
    'PLAYER_NAME': {'operator': '==', 'value': player},
    'GAME_DATE': {'operator': '>', 'value': this_year},
#    'OPPONENT_NAME': {'operator': '==', 'value': 'Miami Heat'},
#    Threes
#    'cse_score_def': {'operator': '>', 'value': 110},
#    'offscreen_cse_score_def': {'operator': '>', 'value': 100},
#    'pnp_cse_score_def': {'operator': '>', 'value': 100},
#    'pue_score_def': {'operator': '>', 'value': 100},
 #   'iso_pue_score_def': {'operator': '>', 'value': 100},
#    'corner_score_def': {'operator': '>', 'value': 100},
    #'open_defense_rating': {'operator': '==', 'value': "Below Average"}

}

# OR filters
or_filters = [
    #{'open_defense_rating': {'operator': '==', 'value': "Elite"}},
    {'open_defense_rating': {'operator': '==', 'value': "Below Average"}},
    {'open_defense_rating': {'operator': '==', 'value': "Poor"}}
]

# Filter with conditions
filtered_df = filter_dataframe(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1, 
                             filters=filters,
                             or_filters=or_filters
                              )

In [874]:
filtered_df[['PLAYER_NAME',"GAME_DATE",'OPPONENT_NAME','PTS','FG3M','FG3A','open_defense_score','open_defense_rating','cse_score_def','offscreen_cse_score_def',
             'pnp_cse_score_def','pue_score_def','iso_pue_score_def','corner_score_def']].head(10)

,PLAYER_NAME,GAME_DATE,OPPONENT_NAME,PTS,FG3M,FG3A,open_defense_score,open_defense_rating,cse_score_def,offscreen_cse_score_def,pnp_cse_score_def,pue_score_def,iso_pue_score_def,corner_score_def
481,Tyrese Maxey,2025-01-31,Denver Nuggets,42,6,11,0.483452,Poor,143.916451,108.132296,130.343967,111.204582,92.053135,2.752529
941,Tyrese Maxey,2025-01-28,Los Angeles Lakers,43,4,11,0.211711,Below Average,126.409914,105.643394,120.595231,94.573433,89.390101,97.714792
1412,Tyrese Maxey,2025-01-25,Chicago Bulls,31,5,13,0.407202,Below Average,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201
2063,Tyrese Maxey,2025-01-21,Denver Nuggets,28,3,10,0.483452,Poor,143.916451,108.132296,130.343967,111.204582,92.053135,2.752529
2453,Tyrese Maxey,2025-01-18,Indiana Pacers,28,2,7,0.308688,Below Average,91.471655,102.357243,110.378227,78.360845,97.321227,119.046894
3660,Tyrese Maxey,2025-01-10,New Orleans Pelicans,30,2,8,0.408965,Poor,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3953,Tyrese Maxey,2025-01-08,Washington Wizards,29,3,14,0.814832,Poor,105.817289,92.407910,128.478624,58.736195,106.509922,55.050587
5269,Tyrese Maxey,2024-12-30,Portland Trail Blazers,23,3,7,0.156318,Below Average,94.393152,118.704245,104.171001,106.417630,111.045211,88.769071
5594,Tyrese Maxey,2024-12-28,Utah Jazz,32,5,13,0.539214,Poor,79.000118,123.219759,95.378719,72.789754,108.193033,79.135219
7478,Tyrese Maxey,2024-12-13,Indiana Pacers,22,3,5,0.308688,Below Average,91.471655,102.357243,110.378227,78.360845,97.321227,119.046894


In [875]:
gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_NAME']==player)&(((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_FG3A_Diff_Current']>0.7)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']>this_year))|
                                                      ((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_FG3A_Diff_Prior']>0.2)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']<this_year)))][["PLAYER_NAME",
                                                      "GAME_DATE",'OPPONENT_NAME','FG3M','FG3A','MIN', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg','MIN_60G_Mavg' ,
 'open_defense_score','open_defense_rating','cse_score_def','offscreen_cse_score_def',
             'pnp_cse_score_def','pue_score_def','iso_pue_score_def','corner_score_def']].head(10)

,PLAYER_NAME,GAME_DATE,OPPONENT_NAME,FG3M,FG3A,MIN,FG3M_60G_Mavg,FG3A_60G_Mavg,MIN_60G_Mavg,open_defense_score,open_defense_rating,cse_score_def,offscreen_cse_score_def,pnp_cse_score_def,pue_score_def,iso_pue_score_def,corner_score_def
481,Tyrese Maxey,2025-01-31,Denver Nuggets,6,11,39.083333,3.183333,9.150000,37.959583,0.483452,Poor,143.916451,108.132296,130.343967,111.204582,92.053135,2.752529
941,Tyrese Maxey,2025-01-28,Los Angeles Lakers,4,11,35.950000,3.166667,9.166667,37.994056,0.211711,Below Average,126.409914,105.643394,120.595231,94.573433,89.390101,97.714792
1412,Tyrese Maxey,2025-01-25,Chicago Bulls,5,13,43.060000,3.133333,9.066667,38.091806,0.407202,Below Average,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201
2063,Tyrese Maxey,2025-01-21,Denver Nuggets,3,10,33.900000,3.066667,8.950000,37.809000,0.483452,Poor,143.916451,108.132296,130.343967,111.204582,92.053135,2.752529
2246,Tyrese Maxey,2025-01-19,Milwaukee Bucks,6,15,41.433333,3.033333,8.866667,37.791472,0.129297,Average,102.160482,103.680458,93.621802,116.783398,99.955109,196.395833
3660,Tyrese Maxey,2025-01-10,New Orleans Pelicans,2,8,40.633333,3.033333,8.700000,37.409528,0.408965,Poor,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3953,Tyrese Maxey,2025-01-08,Washington Wizards,3,14,38.558333,3.016667,8.716667,37.436444,0.814832,Poor,105.817289,92.407910,128.478624,58.736195,106.509922,55.050587
4229,Tyrese Maxey,2025-01-06,Phoenix Suns,6,14,43.900000,3.033333,8.650000,37.352139,-0.166961,Average,113.241202,128.988682,123.045250,150.316128,120.259068,153.453511
5594,Tyrese Maxey,2024-12-28,Utah Jazz,5,13,40.650000,3.066667,8.666667,37.645417,0.539214,Poor,79.000118,123.219759,95.378719,72.789754,108.193033,79.135219
7932,Tyrese Maxey,2024-12-08,Chicago Bulls,3,12,42.433333,2.833333,8.400000,37.507500,0.407202,Below Average,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201


# Points

In [918]:
player= "Miles Bridges"

In [919]:


filters = {
    'PLAYER_NAME': {'operator': '==', 'value': player},
    'GAME_DATE': {'operator': '>', 'value': this_year},
#    'PTS_60G_Mavg': {'operator': '>', 'value': 18},
#    Points
#   'penetrator_score_def': {'operator': '>', 'value': 100},
 #   'pure_score_def': {'operator': '>', 'value': 110},
#    'rim_runner_score_def': {'operator': '>', 'value': 100},
#    'Cluster_current_rank': {'operator': '>', 'value': 100},
#    'Cluster_prior_rank': {'operator': '>', 'value': 100},
#  'OVERALL_DEF_SCORE_PRBallHandler_def': {'operator': '>', 'value': 50},
#   'OVERALL_DEF_SCORE_Isolation_def': {'operator': '>', 'value': 50},
#    'OVERALL_DEF_SCORE_Offscreen_def': {'operator': '>', 'value': 50},
#    'OVERALL_DEF_SCORE_Handoff_def': {'operator': '>', 'value': 50},
#   'OVERALL_DEF_SCORE_OffRebound_def': {'operator': '>', 'value': 50},
#    'OVERALL_DEF_SCORE_PRRollMan_def': {'operator': '>', 'value': 50},
 #   'OVERALL_DEF_SCORE_Postup_def': {'operator': '>', 'value': 50},        
#    'OVERALL_DEF_SCORE_Spotup_def': {'operator': '>', 'value': 50},      
 #  'OVERALL_DEF_SCORE_Misc_def': {'operator': '>', 'value': 50},
#    'OVERALL_DEF_SCORE_Cut_def': {'operator': '>', 'value': 50},
   'OVERALL_DEF_SCORE_Transition_def': {'operator': '>', 'value': 50},

}
# Filter with conditions
filtered_df = filter_dataframe(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1, 
                             filters=filters, )

In [920]:
filtered_df[['PLAYER_NAME',"GAME_DATE",'OPPONENT_NAME','PTS','MIN', 'PTS_60G_Mavg','MIN_60G_Mavg','PTS_10G_Mavg','MIN_10G_Mavg',
             'penetrator_score_def','pure_score_def','rim_runner_score_def','OVERALL_SCORE_Isolation','OVERALL_DEF_SCORE_Isolation_def',
             'OVERALL_SCORE_PRBallHandler','OVERALL_DEF_SCORE_PRBallHandler_def','OVERALL_SCORE_PRRollMan','OVERALL_DEF_SCORE_PRRollMan_def',
             'OVERALL_SCORE_Transition','OVERALL_DEF_SCORE_Transition_def',
             'open_defense_score','open_defense_rating','cse_score_def','offscreen_cse_score_def',
             'pnp_cse_score_def','pue_score_def','iso_pue_score_def','corner_score_def']].head(10)

,PLAYER_NAME,GAME_DATE,OPPONENT_NAME,PTS,MIN,PTS_60G_Mavg,MIN_60G_Mavg,PTS_10G_Mavg,MIN_10G_Mavg,penetrator_score_def,pure_score_def,rim_runner_score_def,OVERALL_SCORE_Isolation,OVERALL_DEF_SCORE_Isolation_def,OVERALL_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRBallHandler_def,OVERALL_SCORE_PRRollMan,OVERALL_DEF_SCORE_PRRollMan_def,OVERALL_SCORE_Transition,OVERALL_DEF_SCORE_Transition_def,open_defense_score,open_defense_rating,cse_score_def,offscreen_cse_score_def,pnp_cse_score_def,pue_score_def,iso_pue_score_def,corner_score_def
3,Miles Bridges,2025-02-03,Washington Wizards,24,36.700000,19.766667,34.685028,23.2,34.226833,87.090290,110.057722,115.369231,39.7,87.8,47.1,51.6,49.8,77.0,58.8,59.5,0.481754,Poor,105.817289,92.407910,128.478624,58.736195,106.509922,55.050587
191,Miles Bridges,2025-02-01,Denver Nuggets,24,33.916667,19.700000,34.716972,23.2,34.226833,75.170251,77.082905,83.639442,39.7,28.0,47.1,28.1,49.8,48.6,58.8,62.1,-0.229502,Good,143.916451,108.132296,130.343967,111.204582,92.053135,2.752529
685,Miles Bridges,2025-01-29,Brooklyn Nets,23,31.966667,19.450000,34.693361,23.2,34.226833,105.790353,90.357831,133.609109,39.7,38.2,47.1,53.8,49.8,54.8,58.8,60.8,0.329572,Below Average,95.739808,103.332970,109.699145,58.018285,72.902120,103.907983
1480,Miles Bridges,2025-01-25,New Orleans Pelicans,22,28.283333,20.066667,34.835861,23.2,34.226833,107.110357,101.991100,118.199212,39.7,44.6,47.1,48.8,49.8,64.4,58.8,76.8,-0.043775,Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
2674,Miles Bridges,2025-01-17,Chicago Bulls,21,31.623333,20.000000,35.104278,23.2,34.226833,100.490335,105.699413,131.329124,39.7,54.3,47.1,44.3,49.8,49.2,58.8,50.1,0.688852,Poor,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201
3059,Miles Bridges,2025-01-15,Utah Jazz,25,38.916667,20.000000,35.225556,23.2,34.226833,108.410361,115.574358,96.459357,39.7,76.1,47.1,56.9,49.8,53.1,58.8,52.9,0.909730,Poor,79.000118,123.219759,95.378719,72.789754,108.193033,79.135219
3407,Miles Bridges,2025-01-12,Phoenix Suns,21,32.033333,19.933333,35.258611,23.2,34.226833,117.430391,96.399464,119.399204,39.7,26.9,47.1,52.4,49.8,54.5,58.8,54.9,-0.206899,Good,113.241202,128.988682,123.045250,150.316128,120.259068,153.453511
4145,Miles Bridges,2025-01-07,Phoenix Suns,21,36.766667,19.933333,35.321111,23.2,34.226833,117.430391,96.399464,119.399204,39.7,26.9,47.1,52.4,49.8,54.5,58.8,54.9,-0.206899,Good,113.241202,128.988682,123.045250,150.316128,120.259068,153.453511
4684,Miles Bridges,2025-01-03,Detroit Pistons,20,39.155000,20.200000,35.609694,23.2,34.226833,98.220327,92.891151,91.419391,39.7,56.8,47.1,53.0,49.8,46.8,58.8,59.2,0.472087,Poor,108.666310,112.359767,107.184519,86.049903,102.158244,41.287940
5298,Miles Bridges,2024-12-30,Chicago Bulls,31,36.183333,20.283333,35.623778,23.2,34.226833,100.490335,105.699413,131.329124,39.7,54.3,47.1,44.3,49.8,49.2,58.8,50.1,0.688852,Poor,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201


In [922]:
showme = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_NAME']==player)&(((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Cluster_current_rank']>22)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']>this_year))|
                                                      ((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_PTS_Diff_Prior']>0.1)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']<this_year)))][["PLAYER_NAME",
                                                      "GAME_DATE",'OPPONENT_NAME','PTS','MIN', 'PTS_60G_Mavg','MIN_60G_Mavg' ,'Avg_PTS_MIN_Diff_Current','Avg_PTS_MIN_Diff_Prior',
'penetrator_score_def','pure_score_def','rim_runner_score_def',
'OVERALL_SCORE_Isolation','OVERALL_DEF_SCORE_Isolation_def','OVERALL_SCORE_PRBallHandler','OVERALL_DEF_SCORE_PRBallHandler_def','OVERALL_SCORE_PRRollMan','OVERALL_DEF_SCORE_PRRollMan_def',
'OVERALL_SCORE_Transition','OVERALL_DEF_SCORE_Transition_def',
 'open_defense_score','open_defense_rating','cse_score_def','offscreen_cse_score_def',
             'pnp_cse_score_def','pue_score_def','iso_pue_score_def','corner_score_def']].head(20)

showme

,PLAYER_NAME,GAME_DATE,OPPONENT_NAME,PTS,MIN,PTS_60G_Mavg,MIN_60G_Mavg,Avg_PTS_MIN_Diff_Current,Avg_PTS_MIN_Diff_Prior,penetrator_score_def,pure_score_def,rim_runner_score_def,OVERALL_SCORE_Isolation,OVERALL_DEF_SCORE_Isolation_def,OVERALL_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRBallHandler_def,OVERALL_SCORE_PRRollMan,OVERALL_DEF_SCORE_PRRollMan_def,OVERALL_SCORE_Transition,OVERALL_DEF_SCORE_Transition_def,open_defense_score,open_defense_rating,cse_score_def,offscreen_cse_score_def,pnp_cse_score_def,pue_score_def,iso_pue_score_def,corner_score_def
191,Miles Bridges,2025-02-01,Denver Nuggets,24,33.916667,19.700000,34.716972,0.089220,-0.031897,75.170251,77.082905,83.639442,39.7,28.0,47.1,28.1,49.8,48.6,58.8,62.1,-0.229502,Good,143.916451,108.132296,130.343967,111.204582,92.053135,2.752529
1480,Miles Bridges,2025-01-25,New Orleans Pelicans,22,28.283333,20.066667,34.835861,0.029516,-0.009402,107.110357,101.991100,118.199212,39.7,44.6,47.1,48.8,49.8,64.4,58.8,76.8,-0.043775,Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
2674,Miles Bridges,2025-01-17,Chicago Bulls,21,31.623333,20.000000,35.104278,0.061557,-0.027555,100.490335,105.699413,131.329124,39.7,54.3,47.1,44.3,49.8,49.2,58.8,50.1,0.688852,Poor,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201
4345,Miles Bridges,2025-01-05,Cleveland Cavaliers,11,26.485000,19.916667,35.388611,0.042285,-0.024210,98.000327,108.407731,86.749422,39.7,52.9,47.1,33.3,49.8,40.1,58.8,33.8,-0.151536,Good,95.612624,96.073198,94.039079,143.151594,129.292368,132.121409
4684,Miles Bridges,2025-01-03,Detroit Pistons,20,39.155000,20.200000,35.609694,0.003818,0.003980,98.220327,92.891151,91.419391,39.7,56.8,47.1,53.0,49.8,46.8,58.8,59.2,0.472087,Poor,108.666310,112.359767,107.184519,86.049903,102.158244,41.287940
5298,Miles Bridges,2024-12-30,Chicago Bulls,31,36.183333,20.283333,35.623778,0.061557,-0.027555,100.490335,105.699413,131.329124,39.7,54.3,47.1,44.3,49.8,49.2,58.8,50.1,0.688852,Poor,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201
6727,Miles Bridges,2024-12-20,Philadelphia 76ers,15,28.733333,19.950000,35.734056,0.019636,-0.008752,81.660272,99.682780,130.229132,39.7,61.8,47.1,32.2,49.8,79.2,58.8,59.2,0.158804,Below Average,112.651572,98.054471,120.172725,82.612676,98.784179,59.179381
7125,Miles Bridges,2024-12-16,Philadelphia 76ers,24,28.230000,20.233333,35.958222,0.019636,-0.008752,81.660272,99.682780,130.229132,39.7,61.8,47.1,32.2,49.8,79.2,58.8,59.2,0.158804,Below Average,112.651572,98.054471,120.172725,82.612676,98.784179,59.179381
7454,Miles Bridges,2024-12-13,Chicago Bulls,14,24.616667,20.300000,36.187167,0.061557,-0.027555,100.490335,105.699413,131.329124,39.7,54.3,47.1,44.3,49.8,49.2,58.8,50.1,0.688852,Poor,117.184381,125.206745,112.404992,88.301652,99.842090,183.043201
11025,Miles Bridges,2024-11-17,Cleveland Cavaliers,19,26.816667,20.600000,36.490500,0.042285,-0.024210,98.000327,108.407731,86.749422,39.7,52.9,47.1,33.3,49.8,40.1,58.8,33.8,-0.151536,Good,95.612624,96.073198,94.039079,143.151594,129.292368,132.121409


In [640]:
showme[showme['PTS']>14]['GAME_DATE'].count()/showme['GAME_DATE'].count()

0.9

In [641]:
filterd_df_pts = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['OPPONENT_NAME']=='New Orleans Pelicans')&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Cluster_Pts']==2)][["PLAYER_NAME",
                                                      "GAME_DATE",'OPPONENT_NAME','PTS','MIN', 'PTS_60G_Mavg','MIN_60G_Mavg' ,'Avg_PTS_MIN_Diff_Current','Avg_PTS_MIN_Diff_Prior',
'penetrator_score_def','pure_score_def','rim_runner_score_def',
'OVERALL_SCORE_Isolation','OVERALL_DEF_SCORE_Isolation_def','OVERALL_SCORE_PRBallHandler','OVERALL_DEF_SCORE_PRBallHandler_def','OVERALL_SCORE_PRRollMan','OVERALL_DEF_SCORE_PRRollMan_def',
'OVERALL_SCORE_Transition','OVERALL_DEF_SCORE_Transition_def',
 'open_defense_score','open_defense_rating','cse_score_def','offscreen_cse_score_def',
             'pnp_cse_score_def','pue_score_def','iso_pue_score_def','corner_score_def']].head(20)

filterd_df_pts

,PLAYER_NAME,GAME_DATE,OPPONENT_NAME,PTS,MIN,PTS_60G_Mavg,MIN_60G_Mavg,Avg_PTS_MIN_Diff_Current,Avg_PTS_MIN_Diff_Prior,penetrator_score_def,pure_score_def,rim_runner_score_def,OVERALL_SCORE_Isolation,OVERALL_DEF_SCORE_Isolation_def,OVERALL_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRBallHandler_def,OVERALL_SCORE_PRRollMan,OVERALL_DEF_SCORE_PRRollMan_def,OVERALL_SCORE_Transition,OVERALL_DEF_SCORE_Transition_def,open_defense_score,open_defense_rating,cse_score_def,offscreen_cse_score_def,pnp_cse_score_def,pue_score_def,iso_pue_score_def,corner_score_def
441,Jrue Holiday,2025-01-31,New Orleans Pelicans,4,29.550000,11.050000,30.651389,0.003972,-0.004479,107.110357,101.9911,118.199212,47.7,44.6,43.4,48.8,40.0,64.4,39.8,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
473,Al Horford,2025-01-31,New Orleans Pelicans,4,25.498333,8.716667,26.495056,0.003972,-0.004479,107.110357,101.9911,118.199212,0.0,44.6,0.0,48.8,32.4,64.4,37.7,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
839,Naji Marshall,2025-01-29,New Orleans Pelicans,5,15.050000,8.983333,22.496944,0.003972,-0.004479,107.110357,101.9911,118.199212,49.9,44.6,42.3,48.8,18.0,64.4,50.1,76.8,NaN,NaN,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
1195,Chris Boucher,2025-01-27,New Orleans Pelicans,14,18.350000,9.600000,16.568472,0.003972,-0.004479,107.110357,101.9911,118.199212,0.0,44.6,0.0,48.8,37.3,64.4,41.9,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
1584,Jaylen Wells,2025-01-24,New Orleans Pelicans,8,29.640000,11.863636,26.054811,0.003972,-0.004479,107.110357,101.9911,118.199212,36.5,44.6,50.7,48.8,0.0,64.4,51.2,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3068,Naji Marshall,2025-01-15,New Orleans Pelicans,7,30.266667,8.800000,22.251139,0.003972,-0.004479,107.110357,101.9911,118.199212,49.9,44.6,42.3,48.8,18.0,64.4,50.1,76.8,NaN,NaN,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3411,Jrue Holiday,2025-01-12,New Orleans Pelicans,8,31.800000,11.650000,31.150306,0.003972,-0.004479,107.110357,101.9911,118.199212,47.7,44.6,43.4,48.8,40.0,64.4,39.8,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3412,Al Horford,2025-01-12,New Orleans Pelicans,11,22.718333,9.066667,26.984444,0.003972,-0.004479,107.110357,101.9911,118.199212,0.0,44.6,0.0,48.8,32.4,64.4,37.7,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3666,Guerschon Yabusele,2025-01-10,New Orleans Pelicans,6,31.455000,9.972222,24.958704,0.003972,-0.004479,107.110357,101.9911,118.199212,0.0,44.6,0.0,48.8,57.2,64.4,51.7,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467
3862,Toumani Camara,2025-01-08,New Orleans Pelicans,15,29.350000,9.100000,28.738889,0.003972,-0.004479,107.110357,101.9911,118.199212,31.7,44.6,0.0,48.8,43.0,64.4,36.0,76.8,0.085506,Below Average,131.063361,105.543253,127.056149,123.263357,115.239442,137.626467


In [521]:
(filterd_df_pts['PTS']-filterd_df_pts['PTS_60G_Mavg']).round(1).mean()

0.38

In [522]:
filterd_df_pts = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_NAME']==player)&(((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_PTS_MIN_Diff_Current']>0)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']>this_year))|
                                                      ((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_PTS_MIN_Diff_Prior']>0)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']<this_year)))].head(20)
filterd_df_pts['PTS_DIFF'] = (filterd_df_pts['PTS']-filterd_df_pts['PTS_60G_Mavg']).round(1)
filterd_df_pts[['PLAYER_NAME','GAME_DATE','OPPONENT_NAME','PTS','PTS_60G_Mavg','PTS_DIFF','MIN']]

,PLAYER_NAME,GAME_DATE,OPPONENT_NAME,PTS,PTS_60G_Mavg,PTS_DIFF,MIN
3259,Isaiah Hartenstein,2025-01-12,Washington Wizards,10,10.066667,-0.1,21.661667
5395,Isaiah Hartenstein,2024-12-29,Memphis Grizzlies,12,9.816667,2.2,25.221667
6090,Isaiah Hartenstein,2024-12-23,Washington Wizards,16,9.600000,6.4,27.100000
7675,Isaiah Hartenstein,2024-12-10,Dallas Mavericks,10,9.150000,0.8,30.296667
8350,Isaiah Hartenstein,2024-12-05,Toronto Raptors,2,9.016667,-7.0,27.850000
8760,Isaiah Hartenstein,2024-12-03,Utah Jazz,4,9.166667,-5.2,22.231667
15787,Isaiah Hartenstein,2024-04-14,Chicago Bulls,8,8.266667,-0.3,30.901667
16405,Isaiah Hartenstein,2024-04-09,Chicago Bulls,11,8.016667,3.0,28.350000
16919,Isaiah Hartenstein,2024-04-07,Milwaukee Bucks,18,7.900000,10.1,28.850000
17223,Isaiah Hartenstein,2024-04-05,Chicago Bulls,10,7.650000,2.3,29.500000


# Rebs

In [926]:
player= "Jalen Duren"

In [927]:
#gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.head(5)

In [928]:
filterd_df_reb = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_NAME']==player)&(((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_Reb_Diff_Current']>.5)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']>this_year))|
                                                      ((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_Reb_Diff_Prior']>.2)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']<this_year)))].head(20)
filterd_df_reb[['PLAYER_NAME','OPPONENT_NAME','GAME_DATE','REB','REB_60G_Mavg','Avg_Reb_Diff_Current','Avg_Reb_Diff_Prior']]

,PLAYER_NAME,OPPONENT_NAME,GAME_DATE,REB,REB_60G_Mavg,Avg_Reb_Diff_Current,Avg_Reb_Diff_Prior
674,Jalen Duren,Indiana Pacers,2025-01-29,10,10.200000,1.195500,-0.020270
2746,Jalen Duren,Indiana Pacers,2025-01-16,17,10.100000,1.195500,-0.020270
3805,Jalen Duren,Golden State Warriors,2025-01-09,12,10.116667,1.509730,0.015075
4663,Jalen Duren,Charlotte Hornets,2025-01-03,14,10.366667,0.675349,0.508904
5474,Jalen Duren,Denver Nuggets,2024-12-28,7,10.450000,1.012857,0.095932
6363,Jalen Duren,Los Angeles Lakers,2024-12-23,9,10.600000,0.517317,0.573077
9242,Jalen Duren,Indiana Pacers,2024-11-29,12,11.083333,1.195500,-0.020270
10451,Jalen Duren,Charlotte Hornets,2024-11-21,9,11.383333,0.675349,0.508904
11179,Jalen Duren,Washington Wizards,2024-11-17,10,11.350000,0.527317,2.443881
12875,Jalen Duren,Charlotte Hornets,2024-11-06,3,11.400000,0.675349,0.508904


In [535]:
filterd_df_reb = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['OPPONENT_NAME']=='Cleveland Cavaliers')&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Rebounding_Cluster']==4)].head(20)
filterd_df_reb[['PLAYER_NAME','OPPONENT_NAME','GAME_DATE','REB','REB_60G_Mavg','Avg_Reb_Diff_Current','Avg_Reb_Diff_Prior']]

,PLAYER_NAME,OPPONENT_NAME,GAME_DATE,REB,REB_60G_Mavg,Avg_Reb_Diff_Current,Avg_Reb_Diff_Prior
623,Larry Nance Jr.,Cleveland Cavaliers,2025-01-30,5,4.733333,-2.642,0.614286
751,Kel'el Ware,Cleveland Cavaliers,2025-01-29,4,4.806452,-2.642,0.614286
1039,Paul Reed,Cleveland Cavaliers,2025-01-27,0,5.100000,-2.642,0.614286
1507,Steven Adams,Cleveland Cavaliers,2025-01-25,4,4.866667,-2.642,0.614286
1842,Steven Adams,Cleveland Cavaliers,2025-01-22,11,4.896552,-2.642,0.614286
2131,Mason Plumlee,Cleveland Cavaliers,2025-01-20,0,5.683333,-2.642,0.614286
2727,Jaylin Williams,Cleveland Cavaliers,2025-01-16,5,3.566667,-2.642,0.614286
3076,Thomas Bryant,Cleveland Cavaliers,2025-01-14,5,3.550000,-2.642,0.614286
3296,Thomas Bryant,Cleveland Cavaliers,2025-01-12,3,3.550000,-2.642,0.614286
3735,Kelly Olynyk,Cleveland Cavaliers,2025-01-09,0,4.783333,-2.642,0.614286


In [2533]:
(filterd_df_reb['REB'] - filterd_df_reb['REB_60G_Mavg']).mean()

nan

# Ast

In [929]:
player= "Alperen Sengun"

In [930]:



filters = {
    'PLAYER_NAME': {'operator': '==', 'value': player},
    'GAME_DATE': {'operator': '>', 'value': this_year},
#    Ast
    'Avg_Ast_Diff_Current': {'operator': '>', 'value': 0.6},
#    'Avg_Ast_Diff_Prior': {'operator': '>', 'value': 0},
   'ast_shooter_score': {'operator': '>', 'value': 100},
#   'ast_bigs_score': {'operator': '>', 'value': 100}

}

# Filter with conditions
filtered_df = filter_dataframe(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1, 
                             filters=filters)

In [931]:
filtered_df[['PLAYER_NAME','OPPONENT_NAME','GAME_DATE',"AST",'AST_60G_Mavg','ast_shooter_score','ast_bigs_score','Avg_Ast_Diff_Current','Avg_Ast_Diff_Prior']]

,PLAYER_NAME,OPPONENT_NAME,GAME_DATE,AST,AST_60G_Mavg,ast_shooter_score,ast_bigs_score,Avg_Ast_Diff_Current,Avg_Ast_Diff_Prior
2150,Alperen Sengun,Detroit Pistons,2025-01-20,5,4.966667,105.496810,102.874095,0.884091,0.547250
2949,Alperen Sengun,Denver Nuggets,2025-01-15,8,5.016667,102.187971,109.344995,0.855000,0.032188
6463,Alperen Sengun,Toronto Raptors,2024-12-22,5,4.933333,119.840831,116.244842,1.207143,0.354706
12267,Alperen Sengun,Detroit Pistons,2024-11-10,2,4.666667,105.496810,102.874095,0.884091,0.547250


In [932]:
filterd_df_ast = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['PLAYER_NAME']==player)&(((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_Ast_Diff_Current']>.4)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']>this_year))|
                                                      ((gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Avg_Ast_Diff_Prior']>.1)&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['GAME_DATE']<this_year)))].head(20)
filterd_df_ast[['PLAYER_NAME','OPPONENT_NAME','GAME_DATE','AST','AST_60G_Mavg','Avg_Ast_Diff_Current','Avg_Ast_Diff_Prior','ast_shooter_score','ast_bigs_score']]

,PLAYER_NAME,OPPONENT_NAME,GAME_DATE,AST,AST_60G_Mavg,Avg_Ast_Diff_Current,Avg_Ast_Diff_Prior,ast_shooter_score,ast_bigs_score
2150,Alperen Sengun,Detroit Pistons,2025-01-20,5,4.966667,0.884091,0.547250,105.496810,102.874095
2949,Alperen Sengun,Denver Nuggets,2025-01-15,8,5.016667,0.855000,0.032188,102.187971,109.344995
6463,Alperen Sengun,Toronto Raptors,2024-12-22,5,4.933333,1.207143,0.354706,119.840831,116.244842
12267,Alperen Sengun,Detroit Pistons,2024-11-10,2,4.666667,0.884091,0.547250,105.496810,102.874095
21414,Alperen Sengun,Sacramento Kings,2024-03-10,2,4.866667,0.212857,0.441818,93.460634,89.538996
21692,Alperen Sengun,Portland Trail Blazers,2024-03-08,6,4.950000,-0.316842,0.164474,114.713696,107.987668
22018,Alperen Sengun,LA Clippers,2024-03-06,14,4.966667,-0.207368,0.174545,74.214186,64.378153
23240,Alperen Sengun,Oklahoma City Thunder,2024-02-27,6,4.929825,-0.787500,0.596585,82.218643,82.105755
23588,Alperen Sengun,Oklahoma City Thunder,2024-02-25,2,4.910714,-0.787500,0.596585,82.218643,82.105755
25203,Alperen Sengun,Toronto Raptors,2024-02-09,1,5.000000,1.207143,0.354706,119.840831,116.244842


In [560]:
filterd_df_reb = gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1.loc[(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['OPPONENT_NAME']=='Miami Heat')&(gamelogs_ALL_sc_cls_dsc_3sc_reb_ast1['Passing_Cluster']==3)].head(20)
filterd_df_reb[['PLAYER_NAME','OPPONENT_NAME','GAME_DATE','AST','REB_60G_Mavg','Avg_Reb_Diff_Current','Avg_Reb_Diff_Prior']]

,PLAYER_NAME,OPPONENT_NAME,GAME_DATE,AST,REB_60G_Mavg,Avg_Reb_Diff_Current,Avg_Reb_Diff_Prior
314,Chris Paul,Miami Heat,2025-02-01,7,4.116667,0.210000,0.64963
1747,Damian Lillard,Miami Heat,2025-01-23,11,4.583333,0.210000,0.64963
2314,Chris Paul,Miami Heat,2025-01-19,9,4.266667,0.210000,0.64963
2621,Nikola Jokić,Miami Heat,2025-01-17,10,13.016667,0.506512,-0.23250
2922,LeBron James,Miami Heat,2025-01-15,9,7.400000,0.506512,-0.23250
3245,James Harden,Miami Heat,2025-01-13,11,5.750000,0.210000,0.64963
4155,Dennis Schröder,Miami Heat,2025-01-07,7,3.083333,NaN,NaN
4811,T.J. McConnell,Miami Heat,2025-01-02,4,2.683333,NaN,NaN
4866,Tyrese Haliburton,Miami Heat,2025-01-02,15,3.850000,0.210000,0.64963
5031,Dejounte Murray,Miami Heat,2025-01-01,7,6.116667,0.210000,0.64963


# Top Matchups

In [3030]:
def analyze_matchups_with_weighted_grade(df):
    def calculate_weighted_score(row):
        def get_weight(value, baseline):
            diff = (value - baseline) / baseline
            # Keep the sign but apply power to absolute value
            return abs(diff) ** 1.5 * (1 if diff >= 0 else -1)
        
        # These should not be inside get_weight function
        advantages = []
        total_weighted_advantage = 0
        
        # Primary Metrics (100 baseline)
        primary_metrics = {
            'Penetrator': ('penetrator_score', 'penetrator_score_def'),
            'Pure Score': ('pure_score', 'pure_score_def'),
            'Rim Runner': ('rim_runner_score', 'rim_runner_score_def')
        }
        
        # Calculate primary metrics advantages
        for metric_name, (off_col, def_col) in primary_metrics.items():
            if row[off_col] == 0:
                continue
                
            advantage = ((row[def_col] - 100)/100)
            weight = get_weight(row[off_col], 100)
            weighted_advantage = abs(weight) * advantage
            
            advantages.append({
                'type': metric_name,
                'offensive_rating': row[off_col],
                'defensive_rating': row[def_col],
                'raw_advantage': advantage,
                'weight': weight,
                'weighted_advantage': weighted_advantage,
                'baseline': 100,
                'is_used': True
            })

        # Play Type Metrics (50 baseline)
        play_types = {
            'Cut': ('OVERALL_SCORE_Cut', 'OVERALL_DEF_SCORE_Cut_def'),
            'Handoff': ('OVERALL_SCORE_Handoff', 'OVERALL_DEF_SCORE_Handoff_def'),
            'Isolation': ('OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def'),
            'Off Screen': ('OVERALL_SCORE_OffScreen', 'OVERALL_DEF_SCORE_OffScreen_def'),
            'P&R Ball Handler': ('OVERALL_SCORE_PRBallHandler', 'OVERALL_DEF_SCORE_PRBallHandler_def'),
            'P&R Roll Man': ('OVERALL_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_PRRollMan_def'),
            'Post Up': ('OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def'),
            'Spot Up': ('OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def'),
            'Transition': ('OVERALL_SCORE_Transition', 'OVERALL_DEF_SCORE_Transition_def')
        }
        
        for play_type, (off_col, def_col) in play_types.items():
            # Skip if offensive rating is 0 (unused play type)
            if row[off_col] == 0:
                continue
                
            advantage = ((row[def_col] - 50)/50)
            weight = get_weight(row[off_col], 50)  # Using the new weight function
            weighted_advantage = abs(weight) * advantage
            
            advantages.append({
                'type': play_type,
                'offensive_rating': row[off_col],
                'defensive_rating': row[def_col],
                'raw_advantage': advantage,
                'weight': weight,
                'weighted_advantage': weighted_advantage,
                'baseline': 50,
                'is_used': True
            })

        # Calculate overall grade
        used_advantages = [adv['weighted_advantage'] for adv in advantages]
        if used_advantages:
            total_weighted_advantage = sum(used_advantages)
            max_possible_advantage = 100
            overall_grade = 50 + (total_weighted_advantage / max_possible_advantage) * 50
            overall_grade = min(100, max(0, overall_grade))
        else:
            overall_grade = 0
            total_weighted_advantage = 0

        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'total_weighted_advantage': total_weighted_advantage,
            'overall_grade': overall_grade,
            'min_avg': row['MIN_60G_Mavg'],
            'min_10avg': row['MIN_10G_Mavg'],
            'pts_avg': row['PTS_60G_Mavg'],
            'pts_10avg': row['PTS_10G_Mavg'],
            'num_used_play_types': len(advantages)
        }

    # Create results list and return it
    results = []
    for _, row in df.iterrows():
        analysis = calculate_weighted_score(row)
        results.append(analysis)
    
    return results  # Make sure to return the results!

def create_detailed_df(matchups):
    rows = []
    for m in matchups:
        for adv in m['advantages']:
            rows.append({
                'player': m['player'],
                'team': m['team'],
                'opponent': m['opponent'],
                'pts_avg': m['pts_avg'],
                'overall_grade': m['overall_grade'],
                'matchup_type': adv['type'],
                'offensive_rating': adv['offensive_rating'],
                'defensive_rating': adv['defensive_rating'],
                'raw_advantage': adv['raw_advantage'],
                'player_strength_weight': adv['weight'],
                'weighted_advantage': adv['weighted_advantage'],
                'baseline': adv['baseline'],
                'is_used_play_type': adv['is_used']
            })
    return pd.DataFrame(rows)

def create_summary_df(matchups):
    summary_data = []
    for m in matchups:
        player_data = {
            'player': m['player'],
            'team': m['team'],
            'opponent': m['opponent'],
            'min_avg': m['min_avg'],
            'min_L10avg': m['min_10avg'],
            'pts_avg': m['pts_avg'],
            'pts_L10avg': m['pts_10avg'],
            'overall_grade': m['overall_grade'],
            'num_used_play_types': m['num_used_play_types']
        }
        
        # Initialize all play type columns to 0
        for adv in m['advantages']:
            col_name = f"{adv['type']}_weighted_grade"
            player_data[col_name] = adv['weighted_advantage']
            
        summary_data.append(player_data)
    
    df = pd.DataFrame(summary_data)
    return df.sort_values('overall_grade', ascending=False)

# Create both DataFrames
matchups = analyze_matchups_with_weighted_grade(pts_scores_df)
detailed_df = create_detailed_df(matchups)
summary_df = create_summary_df(matchups)




In [3031]:
#summary_df.head(15)

In [3032]:
summary_df.loc[(summary_df['min_L10avg']>23)&(summary_df['pts_avg']>10)].head(15)

,player,team,opponent,min_avg,min_L10avg,pts_avg,pts_L10avg,overall_grade,num_used_play_types,Penetrator_weighted_grade,Pure Score_weighted_grade,Rim Runner_weighted_grade,Cut_weighted_grade,Handoff_weighted_grade,Isolation_weighted_grade,Off Screen_weighted_grade,P&R Ball Handler_weighted_grade,P&R Roll Man_weighted_grade,Post Up_weighted_grade,Spot Up_weighted_grade,Transition_weighted_grade
87,De'Aaron Fox,Sacramento Kings,Chicago Bulls,36.95,37.46,26.45,27.3,50.188814,11,0.027678,0.135213,3.611920e-03,0.014535,0.110302,0.036937,0.038738,-0.087271,NaN,0.064110,-0.003750,0.037524
119,Luguentz Dort,Oklahoma City Thunder,Washington Wizards,29.19,30.74,10.65,8.7,50.183806,11,-0.014691,0.011794,8.106145e-03,0.007702,-0.036597,0.271609,-0.029247,-0.000810,0.147236,NaN,0.000377,0.002133
135,Mark Williams,Charlotte Hornets,Phoenix Suns,24.04,23.09,12.50,13.6,50.178438,7,0.048681,-0.038716,1.780764e-01,0.171218,NaN,NaN,NaN,NaN,0.003472,-0.005821,NaN,-0.000034
86,Domantas Sabonis,Sacramento Kings,Chicago Bulls,35.86,36.24,19.43,19.0,50.176878,10,0.000775,0.013977,2.301120e-01,0.087356,NaN,0.035202,NaN,-0.000023,-0.016095,0.001031,-0.004372,0.005793
85,DeMar DeRozan,Sacramento Kings,Chicago Bulls,36.73,35.12,23.58,17.1,50.174660,11,0.010843,0.158546,3.395268e-02,0.052271,0.053881,0.041094,0.004508,-0.035916,NaN,0.029286,-0.000894,0.001749
118,Shai Gilgeous-Alexander,Oklahoma City Thunder,Washington Wizards,34.04,35.02,30.02,33.4,50.143395,11,-0.106435,0.071470,7.614684e-04,0.030048,-0.000259,0.292526,-0.000679,-0.013046,NaN,-0.001101,0.000151,0.013355
20,Patrick Williams,Chicago Bulls,Sacramento Kings,27.83,25.56,10.62,10.5,50.136258,9,0.001015,-0.008721,-3.236798e-03,NaN,-0.002319,0.046883,NaN,0.082924,0.092793,NaN,0.077897,-0.014719
35,Jarrett Allen,Cleveland Cavaliers,Indiana Pacers,30.61,27.42,15.75,15.7,50.135486,9,0.014211,0.039828,8.523264e-02,-0.032397,NaN,0.010392,NaN,NaN,0.131691,-0.006187,0.018332,0.009868
34,Donovan Mitchell,Cleveland Cavaliers,Indiana Pacers,32.92,30.02,23.67,21.3,50.133928,10,0.016759,0.071027,5.966255e-03,NaN,0.004092,0.006305,0.031202,0.024653,0.086250,NaN,0.000164,0.021439
27,Jaylen Brown,Boston Celtics,New Orleans Pelicans,34.53,35.21,23.60,22.9,50.133797,11,0.026531,0.023940,1.060971e-02,-0.031448,0.042779,-0.010937,-0.000907,-0.002118,NaN,0.001366,-0.010215,0.217994


In [3033]:
summary_df.loc[(summary_df['min_L10avg']>23)&(summary_df['pts_avg']>10)].tail(15)

,player,team,opponent,min_avg,min_L10avg,pts_avg,pts_L10avg,overall_grade,num_used_play_types,Penetrator_weighted_grade,Pure Score_weighted_grade,Rim Runner_weighted_grade,Cut_weighted_grade,Handoff_weighted_grade,Isolation_weighted_grade,Off Screen_weighted_grade,P&R Ball Handler_weighted_grade,P&R Roll Man_weighted_grade,Post Up_weighted_grade,Spot Up_weighted_grade,Transition_weighted_grade
83,Michael Porter Jr.,Denver Nuggets,Dallas Mavericks,33.11,31.90,18.27,19.6,49.839170,12,0.004013,0.000116,-0.008316,-0.095961,-0.071931,-0.048252,0.017098,0.000813,-0.016429,-0.062404,-0.040408,-0.000000
2,OG Anunoby,New York Knicks,Milwaukee Bucks,35.93,36.67,15.72,16.8,49.829988,11,0.008835,-0.005286,-0.014128,-0.048432,-0.017708,-0.287180,0.030025,0.011604,NaN,-0.029251,0.001952,0.009545
1,Karl-Anthony Towns,New York Knicks,Milwaukee Bucks,33.50,34.85,23.45,24.8,49.827302,12,0.013157,-0.000848,-0.141839,-0.016381,-0.046990,-0.000963,0.000410,0.012575,-0.098009,-0.070265,0.001341,0.002415
7,Kyrie Irving,Dallas Mavericks,Denver Nuggets,36.05,35.91,25.22,24.8,49.826164,12,-0.056605,-0.093577,-0.010859,0.001729,-0.000424,-0.105001,-0.022233,-0.059345,0.005414,-0.021555,0.002439,0.012344
81,Nikola Jokić,Denver Nuggets,Dallas Mavericks,36.44,36.25,28.75,30.2,49.818096,12,0.010441,0.000226,-0.049373,-0.141815,-0.003225,-0.000000,0.016106,0.000430,-0.041936,-0.135695,-0.003408,-0.015558
55,Corey Kispert,Washington Wizards,Oklahoma City Thunder,29.57,27.29,13.67,13.6,49.815833,11,-0.021399,-0.002515,-0.031913,-0.001317,-0.021379,-0.031278,0.000195,-0.175652,-0.031526,NaN,-0.040832,-0.010720
101,Pascal Siakam,Indiana Pacers,Cleveland Cavaliers,32.31,29.76,20.35,19.0,49.815689,12,-0.000004,0.003666,-0.043990,-0.002861,-0.010870,-0.000928,-0.036708,-0.134426,-0.025990,-0.048838,-0.002558,-0.065115
100,Myles Turner,Indiana Pacers,Cleveland Cavaliers,29.12,30.81,15.80,15.7,49.810187,10,-0.000449,0.001488,-0.071153,-0.015143,-0.119581,-0.017371,NaN,NaN,-0.114551,-0.003021,-0.002245,-0.037600
92,Brandon Ingram,New Orleans Pelicans,Boston Celtics,32.58,33.46,20.20,20.3,49.800922,12,-0.019052,-0.071055,-0.001326,0.001275,0.027544,-0.014054,-0.012165,-0.000349,-0.116829,-0.016028,-0.152786,-0.023330
97,Trey Murphy III,New Orleans Pelicans,Boston Celtics,32.74,34.72,17.55,23.8,49.778229,11,-0.010444,-0.003095,-0.001189,0.010663,0.045859,-0.040583,-0.021340,-0.025403,-0.014144,NaN,-0.253381,-0.130487


In [3034]:
#summary_df.loc[(summary_df['player']=='RJ Barrett')].head(15)

In [3035]:
gl_3s_matchups = gl_3s_today_scores_.merge(gl_3pt_closest_def_[['PLAYER_ID','open_defense_score', 'tight_defense_score', 'open_defense_rating',
       'tight_defense_rating', 'defense3_score',]])

In [3036]:
#gl_3s_matchups.to_csv('gl_3s_matchups.csv')

In [3037]:
def analyze_matchups_with_weighted_grade(df):
    def calculate_weighted_score(row):
        def get_weight(value, baseline):
            diff = (value - baseline) / max(abs(baseline), 1)  # Prevent division by zero
            return abs(diff) ** 1.5 * (1 if diff >= 0 else -1)

        avg_fg3a = df['FG3A_60G_Mavg'].mean()
        advantages = []
        total_weighted_advantage = 0
        
        # Primary Metrics with their respective baselines and fixed weights
        primary_metrics = {
            'CatchShoot': ('cse_score', 'cse_score_def', 100, None),
            'Offscreen': ('offscreen_cse_score', 'offscreen_cse_score_def', 100, None),
            'PickPop': ('pnp_cse_score', 'pnp_cse_score_def', 100, None),
            'Pullup': ('pue_score', 'pue_score_def', 100, None),
            'ISO': ('iso_pue_score', 'iso_pue_score_def', 100, None),
            'Corner': ('corner_score', 'corner_score_def', 100, None),
            #'OpenDefense': ('open_defense_score', None, 0, .1),  # Fixed weight for open_defense_score
            #'Defense3': ('defense3_score', None, 0, .05)          # Fixed weight for defense3_score
        }
        
        # Calculate primary metrics advantages
        for metric_name, (off_col, def_col, baseline, fixed_weight) in primary_metrics.items():
            if row[off_col] == 0:
                continue
            
            if fixed_weight is None:
                # Use standard advantage and weight calculation
                if baseline == 100:
                    advantage =  ((row[def_col] - baseline) / 10) if def_col else row[off_col]
                elif baseline == 0:
                    advantage = row[off_col]
                weight = get_weight(row[off_col], baseline)
            else:
                # Fixed weight calculation for special metrics
                advantage = row[off_col]
                weight = fixed_weight
            
            weighted_advantage = abs(weight) * advantage
            
            advantages.append({
                'type': metric_name,
                'offensive_rating': row[off_col],
                'defensive_rating': row[def_col] if def_col else None,
                'raw_advantage': advantage,
                'weight': weight,
                'weighted_advantage': weighted_advantage,
                'baseline': baseline,
                'is_used': True
            })
        
        # Calculate overall grade
        used_advantages = [adv['weighted_advantage'] for adv in advantages]
        if used_advantages:
            total_weighted_advantage = sum(used_advantages)
            max_possible_advantage = 100
            overall_grade = 50 + (total_weighted_advantage / max_possible_advantage) * 50
            overall_grade = min(100, max(0, overall_grade))
        else:
            overall_grade = 0
            total_weighted_advantage = 0
        # open_defense factor    
        shooting_volume_ratio = ((row['FG3A_60G_Mavg'] / avg_fg3a)*.3) + ((row['FG3A_10G_Mavg'] / avg_fg3a)*.2)/2
        open_defense_modifier = (row['open_defense_score'] * shooting_volume_ratio)
        #open_defense_modifier = row['open_defense_score'] * 0.5  # Scaling factor (adjust as needed)
        overall_grade += open_defense_modifier
        overall_grade = min(100, max(0, overall_grade))

        
        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'total_weighted_advantage': total_weighted_advantage,
            'overall_grade': overall_grade,
            'shooting_volume_ratio': shooting_volume_ratio,
            'min_avg': row['MIN_60G_Mavg'],
            'min_10avg': row['MIN_10G_Mavg'],
            'FG3M_avg': row['FG3M_60G_Mavg'],
            'FG3M_10avg': row['FG3M_10G_Mavg'],
            'FG3A_avg': row['FG3A_60G_Mavg'],
            'FG3A_10avg': row['FG3A_10G_Mavg'],
            'num_used_play_types': len(advantages)
        }

    # Create results list and return it
    results = []
    for _, row in df.iterrows():
        analysis = calculate_weighted_score(row)
        results.append(analysis)
    
    return results  # Make sure to return the results!

def create_detailed_df(matchups):
    rows = []
    for m in matchups:
        for adv in m['advantages']:
            rows.append({
                'player': m['player'],
                'team': m['team'],
                'opponent': m['opponent'],
                #'pts_avg': m['pts_avg'],
                'overall_grade': m['overall_grade'],
                'matchup_type': adv['type'],
                'offensive_rating': adv['offensive_rating'],
                'defensive_rating': adv['defensive_rating'],
                'raw_advantage': adv['raw_advantage'],
                'player_strength_weight': adv['weight'],
                'weighted_advantage': adv['weighted_advantage'],
                'baseline': adv['baseline'],
                'is_used_play_type': adv['is_used']
            })
    return pd.DataFrame(rows)

def create_summary_df(matchups):
    summary_data = []
    for m in matchups:
        player_data = {
            'player': m['player'],
            'team': m['team'],
            'opponent': m['opponent'],
            'min_avg': m['min_avg'],
            'min_L10avg': m['min_10avg'],
            #'pts_avg': m['pts_avg'],
            #'pts_L10avg': m['pts_10avg'],
            'FG3M_avg': m['FG3M_avg'],
            'FG3A_avg': m['FG3A_avg'],
            'FG3M_L10avg': m['FG3M_10avg'],
            'FG3A_L10avg': m['FG3A_10avg'],
            'overall_grade': m['overall_grade'],
            'shooting_volume_ratio': m['shooting_volume_ratio'],
            #'num_used_play_types': m['num_used_play_types']
        }
        
        # Initialize all play type columns to 0
        for adv in m['advantages']:
            col_name = f"{adv['type']}_weighted_grade"
            player_data[col_name] = adv['weighted_advantage']
            
        summary_data.append(player_data)
    
    df = pd.DataFrame(summary_data)
    return df.sort_values('overall_grade', ascending=False)

# Create both DataFrames
matchups = analyze_matchups_with_weighted_grade(gl_3s_matchups)
detailed_df = create_detailed_df(matchups)
summary_df = create_summary_df(matchups)


In [3038]:
summary_df.loc[(summary_df['min_L10avg']>23)&(summary_df['FG3A_avg']>4)].head(15).round(2)

,player,team,opponent,min_avg,min_L10avg,FG3M_avg,FG3A_avg,FG3M_L10avg,FG3A_L10avg,overall_grade,shooting_volume_ratio,CatchShoot_weighted_grade,Offscreen_weighted_grade,PickPop_weighted_grade,Pullup_weighted_grade,ISO_weighted_grade,Corner_weighted_grade
26,Jayson Tatum,Boston Celtics,New Orleans Pelicans,36.12,36.46,3.52,9.35,3.9,10.5,70.59,0.75,-0.05,-0.05,0.01,26.89,11.52,2.68
13,Luka Dončić,Dallas Mavericks,Denver Nuggets,37.21,32.94,3.85,10.35,3.6,9.0,63.15,0.78,0.11,-0.00,0.62,25.76,5.70,-6.26
86,De'Aaron Fox,Sacramento Kings,Chicago Bulls,36.95,37.46,2.37,7.00,1.9,6.4,56.28,0.53,0.47,0.08,0.41,4.40,4.18,2.86
24,Kristaps Porziņģis,Boston Celtics,New Orleans Pelicans,29.06,28.12,1.93,5.33,1.7,5.4,55.59,0.42,-0.04,-0.15,0.06,4.57,1.61,5.21
9,Kyrie Irving,Dallas Mavericks,Denver Nuggets,36.05,35.91,3.10,7.32,3.2,7.6,54.19,0.57,0.03,-0.00,0.11,5.98,2.04,-0.06
87,Malik Monk,Sacramento Kings,Chicago Bulls,28.53,34.59,1.98,6.15,2.5,8.2,53.58,0.52,0.22,0.00,0.06,3.11,2.22,1.39
89,Keegan Murray,Sacramento Kings,Chicago Bulls,35.59,32.13,2.15,6.42,1.8,5.5,53.19,0.48,0.24,0.24,0.29,3.16,0.86,1.34
5,Mikal Bridges,New York Knicks,Milwaukee Bucks,37.36,41.85,2.70,7.18,2.6,7.3,53.01,0.56,0.42,0.84,0.37,0.37,-0.03,3.61
28,Payton Pritchard,Boston Celtics,New Orleans Pelicans,27.53,25.56,2.93,7.08,2.4,6.5,52.65,0.54,-0.09,-0.50,0.04,2.77,1.07,1.44
3,OG Anunoby,New York Knicks,Milwaukee Bucks,35.93,36.67,1.98,5.37,2.1,5.6,52.19,0.42,0.11,0.30,0.06,0.66,-0.17,3.09


In [3039]:
summary_df.loc[(summary_df['min_L10avg']>23)&(summary_df['FG3A_avg']>4)].tail(20).round(2)

,player,team,opponent,min_avg,min_L10avg,FG3M_avg,FG3A_avg,FG3M_L10avg,FG3A_L10avg,overall_grade,shooting_volume_ratio,CatchShoot_weighted_grade,Offscreen_weighted_grade,PickPop_weighted_grade,Pullup_weighted_grade,ISO_weighted_grade,Corner_weighted_grade
42,Franz Wagner,Orlando Magic,Philadelphia 76ers,32.35,36.07,1.50,5.20,1.9,7.0,48.65,0.44,-0.72,-0.01,-0.01,-0.21,-0.47,-1.48
96,Jose Alvarado,New Orleans Pelicans,Boston Celtics,19.96,25.51,1.55,4.15,2.4,6.1,48.64,0.36,0.01,-0.32,-0.22,0.02,0.00,-1.86
62,Keyonte George,Utah Jazz,Brooklyn Nets,31.01,31.50,2.48,7.20,2.3,6.4,48.62,0.54,0.00,0.00,0.19,-1.63,-0.92,0.00
53,Carlton Carrington,Washington Wizards,Oklahoma City Thunder,30.22,32.53,1.60,4.46,2.0,4.4,48.55,0.35,0.02,-0.21,-1.27,-0.62,-0.45,-0.32
39,Kentavious Caldwell-Pope,Orlando Magic,Philadelphia 76ers,31.19,32.61,1.75,4.80,1.9,5.4,48.50,0.39,-0.03,-0.08,-0.01,-1.54,-1.02,-0.25
92,Brandon Ingram,New Orleans Pelicans,Boston Celtics,32.58,33.46,1.68,4.38,2.3,6.2,48.15,0.38,0.05,-0.14,-0.02,0.00,0.03,-3.37
20,Ayo Dosunmu,Chicago Bulls,Sacramento Kings,33.88,34.00,1.78,4.73,1.8,4.7,48.00,0.37,-0.00,0.19,0.23,-1.28,-0.04,-3.44
54,Kyshawn George,Washington Wizards,Oklahoma City Thunder,25.57,25.30,1.20,5.00,1.2,5.0,47.89,0.39,0.01,-0.16,-0.96,-0.71,-2.08,-0.26
98,Myles Turner,Indiana Pacers,Cleveland Cavaliers,29.12,30.81,1.88,4.77,1.7,4.9,47.81,0.37,-0.20,-0.08,-1.10,-2.74,-1.08,0.83
95,Trey Murphy III,New Orleans Pelicans,Boston Celtics,32.74,34.72,3.15,8.38,3.6,9.2,47.70,0.67,0.12,-1.30,-1.41,0.07,0.00,-1.17


In [2548]:
def analyze_prime_matchups(df):
    def calculate_mismatch_score(row):
        advantages = []
        
        # Primary Metrics (100 baseline)
        primary_metrics = {
            'Penetrator': ('penetrator_score', 'penetrator_score_def'),
            'Pure Score': ('pure_score', 'pure_score_def'),
            'Rim Runner': ('rim_runner_score', 'rim_runner_score_def')
        }
        
        for metric_name, (off_col, def_col) in primary_metrics.items():
            if row[off_col] > 100 and row[def_col] > 100:
                advantages.append({
                    'type': metric_name,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'advantage': (row[off_col] - 100) * ((row[def_col] - 100)/100),
                    'baseline': 100
                })
        
        # Play Type Metrics (50 baseline)
        play_types = {
            'Cut': ('OVERALL_SCORE_Cut', 'OVERALL_DEF_SCORE_Cut_def'),
            'Handoff': ('OVERALL_SCORE_Handoff', 'OVERALL_DEF_SCORE_Handoff_def'),
            'Isolation': ('OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def'),
            'Off Screen': ('OVERALL_SCORE_OffScreen', 'OVERALL_DEF_SCORE_OffScreen_def'),
            'P&R Ball Handler': ('OVERALL_SCORE_PRBallHandler', 'OVERALL_DEF_SCORE_PRBallHandler_def'),
            'P&R Roll Man': ('OVERALL_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_PRRollMan_def'),
            'Post Up': ('OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def'),
            'Spot Up': ('OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def'),
            'Transition': ('OVERALL_SCORE_Transition', 'OVERALL_DEF_SCORE_Transition_def')
        }
        
        for play_type, (off_col, def_col) in play_types.items():
            if row[off_col] > 50 and row[def_col] > 50:
                advantages.append({
                    'type': play_type,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'advantage': (row[off_col] - 50) * ((row[def_col] - 50)/50),
                    'baseline': 50
                })
        
        total_advantage = sum(adv['advantage'] for adv in advantages)
        
        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'total_advantage': total_advantage,
            'pts_avg': row['PTS_60G_Mavg'],
            'prime_matchups': len([adv for adv in advantages if 
                                 (adv['baseline'] == 100 and adv['offensive_rating'] > 120 and adv['defensive_rating'] > 110) or
                                 (adv['baseline'] == 50 and adv['offensive_rating'] > 60 and adv['defensive_rating'] > 55)])
        }
    
    results = []
    for _, row in df.iterrows():
        analysis = calculate_mismatch_score(row)
        if analysis['total_advantage'] > 0:
            results.append(analysis)
    
    # Sort by number of prime matchups first, then total advantage
    results.sort(key=lambda x: (x['prime_matchups'], x['total_advantage']), reverse=True)
    return results

# Analyze matchups
matchups = analyze_prime_matchups(pts_scores_df)

print("\nBest Matchup Opportunities (Both Offense & Defense Above Baseline):")
print("-" * 80)
for i, result in enumerate(matchups[:10], 1):
    print(f"\n{i}. {result['player']} ({result['team']} vs {result['opponent']})")
    print(f"Season Average: {result['pts_avg']:.1f} PPG")
    print(f"Prime Matchup Count: {result['prime_matchups']}")
    print("\nFavorable Matchups:")
    # Sort advantages by size of advantage
    sorted_advantages = sorted(result['advantages'], key=lambda x: x['advantage'], reverse=True)
    for adv in sorted_advantages:
        print(f"- {adv['type']}:")
        print(f"  Offense: {adv['offensive_rating']:.1f} vs Defense: {adv['defensive_rating']:.1f}")
        print(f"  Both above {adv['baseline']} baseline")


Best Matchup Opportunities (Both Offense & Defense Above Baseline):
--------------------------------------------------------------------------------

1. Nikola Jokić (Denver Nuggets vs Brooklyn Nets)
Season Average: 28.8 PPG
Prime Matchup Count: 4

Favorable Matchups:
- Rim Runner:
  Offense: 197.5 vs Defense: 134.9
  Both above 100 baseline
- Post Up:
  Offense: 88.1 vs Defense: 83.0
  Both above 50 baseline
- Cut:
  Offense: 75.3 vs Defense: 80.4
  Both above 50 baseline
- P&R Roll Man:
  Offense: 72.6 vs Defense: 55.7
  Both above 50 baseline
- Transition:
  Offense: 55.0 vs Defense: 54.1
  Both above 50 baseline
- Spot Up:
  Offense: 53.5 vs Defense: 54.2
  Both above 50 baseline
- Pure Score:
  Offense: 102.1 vs Defense: 100.9
  Both above 100 baseline

2. Joel Embiid (Philadelphia 76ers vs New Orleans Pelicans)
Season Average: 32.1 PPG
Prime Matchup Count: 4

Favorable Matchups:
- Rim Runner:
  Offense: 178.5 vs Defense: 114.6
  Both above 100 baseline
- Handoff:
  Offense: 63.9

In [2549]:
def create_detailed_df(matchups):
    """Creates a detailed DataFrame including all advantages"""
    rows = []
    for m in matchups:
        for adv in m['advantages']:
            rows.append({
                'player': m['player'],
                'team': m['team'],
                'opponent': m['opponent'],
                'pts_avg': m['pts_avg'],
                'matchup_type': adv['type'],
                'offensive_rating': adv['offensive_rating'],
                'defensive_rating': adv['defensive_rating'],
                'advantage': adv['advantage'],
                'baseline': adv['baseline']
            })
    return pd.DataFrame(rows)

detailed_df = create_detailed_df(matchups)

In [2550]:
detailed_df

,player,team,opponent,pts_avg,matchup_type,offensive_rating,defensive_rating,advantage,baseline
0,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Pure Score,102.12,100.92,0.019504,100
1,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Rim Runner,197.48,134.89,34.010772,100
2,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Cut,75.30,80.40,15.382400,50
3,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,P&R Roll Man,72.60,55.70,2.576400,50
4,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Post Up,88.10,83.00,25.146000,50
...,...,...,...,...,...,...,...,...,...
206,Gary Trent Jr.,Milwaukee Bucks,Orlando Magic,13.00,Handoff,50.90,56.50,0.117000,50
207,Patrick Williams,Chicago Bulls,Washington Wizards,10.62,Spot Up,64.40,50.30,0.086400,50
208,Tristan da Silva,Orlando Magic,Milwaukee Bucks,8.83,Spot Up,53.10,50.60,0.037200,50
209,CJ McCollum,New Orleans Pelicans,Philadelphia 76ers,21.12,Transition,58.40,50.10,0.016800,50


In [2551]:
pts_scores_df

,PLAYER_ID,PLAYER_NAME,TEAM_NAME,Cluster_Pts,OPPONENT_NAME,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,FGA_60G_Mavg,GAMES_IN_WINDOW_10G,MIN_10G_Mavg,PTS_10G_Mavg,FGA_10G_Mavg,penetrator_score,penetrator_score_def,pure_score,pure_score_def,rim_runner_score,rim_runner_score_def,OVERALL_SCORE_Cut,OVERALL_DEF_SCORE_Cut_def,OVERALL_SCORE_Handoff,OVERALL_DEF_SCORE_Handoff_def,OVERALL_SCORE_Isolation,OVERALL_DEF_SCORE_Isolation_def,OVERALL_SCORE_Misc,OVERALL_DEF_SCORE_Misc_def,OVERALL_SCORE_OffRebound,OVERALL_DEF_SCORE_OffRebound_def,OVERALL_SCORE_OffScreen,OVERALL_DEF_SCORE_OffScreen_def,OVERALL_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRBallHandler_def,OVERALL_SCORE_PRRollMan,OVERALL_DEF_SCORE_PRRollMan_def,OVERALL_SCORE_Postup,OVERALL_DEF_SCORE_Postup_def,OVERALL_SCORE_Spotup,OVERALL_DEF_SCORE_Spotup_def,OVERALL_SCORE_Transition,OVERALL_DEF_SCORE_Transition_def,PTS_over_15_60G_Sum,PTS_over_20_60G_Sum,PTS_over_25_60G_Sum,PTS_over_30_60G_Sum,PTS_over_35_60G_Sum,PTS_over_40_60G_Sum,as_of,id
0,204456,T.J. McConnell,Indiana Pacers,1.0,Golden State Warriors,60.0,19.51,11.82,9.65,10.0,18.21,8.5,7.9,185.69,68.69,152.63,85.86,79.50,88.95,50.8,38.7,42.9,39.4,48.3,40.6,43.7,66.7,0.0,69.0,0.0,23.0,61.3,20.0,0.0,34.1,0.0,63.9,37.7,48.5,33.6,8.5,16.0,3.0,2.0,1.0,0.0,0.0,2025-01-10,2025-01-10_204456Golden State Warriors
1,1626167,Myles Turner,Indiana Pacers,4.0,Golden State Warriors,60.0,29.12,15.80,11.42,10.0,30.81,15.7,11.2,57.13,68.69,91.17,85.86,157.90,88.95,58.2,38.7,22.3,39.4,35.9,40.6,45.3,66.7,52.8,69.0,0.0,23.0,0.0,20.0,75.0,34.1,47.2,63.9,45.6,48.5,40.5,8.5,29.0,15.0,6.0,4.0,0.0,0.0,2025-01-10,2025-01-10_1626167Golden State Warriors
2,1627783,Pascal Siakam,Indiana Pacers,5.0,Golden State Warriors,60.0,32.31,20.35,15.07,10.0,29.76,19.0,14.5,98.07,68.69,116.11,85.86,142.02,88.95,47.3,38.7,44.4,39.4,48.0,40.6,49.3,66.7,42.2,69.0,35.7,23.0,22.3,20.0,59.3,34.1,67.9,63.9,54.8,48.5,63.7,8.5,49.0,32.0,16.0,2.0,1.0,0.0,2025-01-10,2025-01-10_1627783Golden State Warriors
3,1629614,Andrew Nembhard,Indiana Pacers,1.0,Golden State Warriors,60.0,27.94,10.05,8.12,10.0,30.03,13.3,10.3,109.76,68.69,109.87,85.86,60.38,88.95,28.1,38.7,42.6,39.4,44.5,40.6,51.6,66.7,28.1,69.0,0.0,23.0,49.5,20.0,0.0,34.1,0.0,63.9,42.4,48.5,49.2,8.5,12.0,2.0,0.0,0.0,0.0,0.0,2025-01-10,2025-01-10_1629614Golden State Warriors
4,1630167,Obi Toppin,Indiana Pacers,4.0,Golden State Warriors,60.0,19.12,10.05,6.90,10.0,17.80,10.4,6.4,84.01,68.69,52.84,85.86,121.94,88.95,52.4,38.7,36.8,39.4,0.0,40.6,50.0,66.7,42.4,69.0,27.6,23.0,29.0,20.0,40.6,34.1,59.8,63.9,45.6,48.5,57.9,8.5,13.0,3.0,0.0,0.0,0.0,0.0,2025-01-10,2025-01-10_1630167Golden State Warriors
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,1629022,Lonnie Walker IV,Brooklyn Nets,0.0,Denver Nuggets,58.0,17.43,9.74,8.48,10.0,16.11,6.5,7.6,72.94,76.47,94.90,83.57,0.00,93.24,0.0,55.6,40.2,37.2,38.2,35.5,0.0,63.2,0.0,33.5,29.9,32.3,42.4,31.0,0.0,59.1,0.0,22.0,50.0,54.1,0.0,63.8,17.0,8.0,1.0,0.0,0.0,0.0,2025-01-10,2025-01-10_1629022Denver Nuggets
102,1629651,Nic Claxton,Brooklyn Nets,3.0,Denver Nuggets,60.0,27.98,10.43,7.60,10.0,29.19,10.6,9.6,62.76,76.47,51.00,83.57,151.11,93.24,68.2,55.6,0.0,37.2,28.6,35.5,45.6,63.2,49.5,33.5,0.0,32.3,0.0,31.0,55.8,59.1,35.5,22.0,27.1,54.1,53.1,63.8,11.0,3.0,0.0,0.0,0.0,0.0,2025-01-10,2025-01-10_1629651Denver Nuggets
103,1629661,Cameron Johnson,Brooklyn Nets,6.0,Denver Nuggets,60.0,29.54,16.10,11.40,10.0,32.72,21.8,13.2,89.61,76.47,106.93,83.57,100.01,93.24,48.6,55.6,69.2,37.2,43.5,35.5,60.1,63.2,40.0,33.5,63.2,32.3,40.0,31.0,27.5,59.1,0.0,22.0,62.6,54.1,58.2,63.8,28.0,17.0,10.0,3.0,1.0,0.0,2025-01-10,2025-01-10_1629661Denver Nuggets
104,1630533,Ziaire Williams,Brooklyn Nets,6.0,Denver Nuggets,60.0,20.33,8.40,7.33,10.0,22.91,9.1,7.6,73.38,76.47,57.75,83.57,73.18,93.24,38.5,55.6,38.8,37.2,0.0,35.5,47.0,63.2,45.2,33.5,47.2,32.3,27.1,31.0,0.0,59.1,

In [2552]:
def analyze_matchups_with_highlights(df):
    def evaluate_mismatch_strength(off_rating, def_rating, baseline):
        """
        Evaluates the strength of a mismatch:
        'STRONG' = Both ratings significantly above baseline
        'GOOD' = Both ratings moderately above baseline
        'NORMAL' = Both ratings just above baseline
        """
        if baseline == 100:
            if off_rating > 140 and def_rating > 120:
                return 'STRONG'
            elif off_rating > 120 and def_rating > 110:
                return 'GOOD'
            elif off_rating > 100 and def_rating > 100:
                return 'NORMAL'
        else:  # baseline 50
            if off_rating > 65 and def_rating > 60:
                return 'STRONG'
            elif off_rating > 57 and def_rating > 55:
                return 'GOOD'
            elif off_rating > 50 and def_rating > 50:
                return 'NORMAL'
        return None

    def calculate_mismatch_score(row):
        advantages = []
        
        # Primary Metrics (100 baseline)
        primary_metrics = {
            'Penetrator': ('penetrator_score', 'penetrator_score_def'),
            'Pure Score': ('pure_score', 'pure_score_def'),
            'Rim Runner': ('rim_runner_score', 'rim_runner_score_def')
        }
        
        # Play Type Metrics (50 baseline)
        play_types = {
            'Cut': ('OVERALL_SCORE_Cut', 'OVERALL_DEF_SCORE_Cut_def'),
            'Handoff': ('OVERALL_SCORE_Handoff', 'OVERALL_DEF_SCORE_Handoff_def'),
            'Isolation': ('OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def'),
            'Off Screen': ('OVERALL_SCORE_OffScreen', 'OVERALL_DEF_SCORE_OffScreen_def'),
            'P&R Ball Handler': ('OVERALL_SCORE_PRBallHandler', 'OVERALL_DEF_SCORE_PRBallHandler_def'),
            'P&R Roll Man': ('OVERALL_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_PRRollMan_def'),
            'Post Up': ('OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def'),
            'Spot Up': ('OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def'),
            'Transition': ('OVERALL_SCORE_Transition', 'OVERALL_DEF_SCORE_Transition_def')
        }
        
        # Check primary metrics
        for metric_name, (off_col, def_col) in primary_metrics.items():
            strength = evaluate_mismatch_strength(row[off_col], row[def_col], 100)
            if strength:
                advantages.append({
                    'type': metric_name,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'strength': strength,
                    'baseline': 100,
                    'advantage': (row[off_col] - 100) * ((row[def_col] - 100)/100)
                })

        # Check play types
        for play_type, (off_col, def_col) in play_types.items():
            strength = evaluate_mismatch_strength(row[off_col], row[def_col], 50)
            if strength:
                advantages.append({
                    'type': play_type,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'strength': strength,
                    'baseline': 50,
                    'advantage': (row[off_col] - 50) * ((row[def_col] - 50)/50)
                })

        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'strong_advantages': len([adv for adv in advantages if adv['strength'] == 'STRONG']),
            'total_advantage': sum(adv['advantage'] for adv in advantages),
            'pts_avg': row['PTS_60G_Mavg']
        }

    results = []
    for _, row in df.iterrows():
        analysis = calculate_mismatch_score(row)
        if analysis['advantages']:  # Only include players with advantages
            results.append(analysis)
    
    # Sort by number of strong advantages first, then total advantage
    results.sort(key=lambda x: (x['strong_advantages'], x['total_advantage']), reverse=True)
    return results

# Analyze matchups
matchups = analyze_matchups_with_highlights(pts_scores_df)

print("\nBest Matchup Opportunities (With Highlighted Strengths):")
print("-" * 80)
for i, result in enumerate(matchups[:10], 1):
    print(f"\n{i}. {result['player']} ({result['team']} vs {result['opponent']})")
    print(f"Season Average: {result['pts_avg']:.1f} PPG")
    print(f"Strong Advantages: {result['strong_advantages']}")
    print("\nMatchup Analysis:")
    
    # Sort advantages by strength and then by advantage score
    sorted_advantages = sorted(result['advantages'], 
                             key=lambda x: (0 if x['strength'] == 'STRONG' else 
                                          1 if x['strength'] == 'GOOD' else 2,
                                          x['advantage']), 
                             reverse=True)
    
    for adv in sorted_advantages:
        strength_indicator = "🔥" if adv['strength'] == 'STRONG' else "✅" if adv['strength'] == 'GOOD' else "·"
        print(f"\n{strength_indicator} {adv['type']}:")
        print(f"  Offense: {adv['offensive_rating']:.1f} vs Defense: {adv['defensive_rating']:.1f}")
        if adv['strength'] == 'STRONG':
            print(f"  STRONG MISMATCH - Exceptional opportunity")
        elif adv['strength'] == 'GOOD':
            print(f"  GOOD MISMATCH - Above average opportunity")


Best Matchup Opportunities (With Highlighted Strengths):
--------------------------------------------------------------------------------

1. Nikola Jokić (Denver Nuggets vs Brooklyn Nets)
Season Average: 28.8 PPG
Strong Advantages: 3

Matchup Analysis:

· Transition:
  Offense: 55.0 vs Defense: 54.1

· Spot Up:
  Offense: 53.5 vs Defense: 54.2

· Pure Score:
  Offense: 102.1 vs Defense: 100.9

✅ P&R Roll Man:
  Offense: 72.6 vs Defense: 55.7
  GOOD MISMATCH - Above average opportunity

🔥 Rim Runner:
  Offense: 197.5 vs Defense: 134.9
  STRONG MISMATCH - Exceptional opportunity

🔥 Post Up:
  Offense: 88.1 vs Defense: 83.0
  STRONG MISMATCH - Exceptional opportunity

🔥 Cut:
  Offense: 75.3 vs Defense: 80.4
  STRONG MISMATCH - Exceptional opportunity

2. Yves Missi (New Orleans Pelicans vs Philadelphia 76ers)
Season Average: 9.2 PPG
Strong Advantages: 2

Matchup Analysis:

· Transition:
  Offense: 55.1 vs Defense: 50.1

🔥 Rim Runner:
  Offense: 153.6 vs Defense: 131.5
  STRONG MISMATCH 

In [2553]:
def analyze_matchups_with_advantages(df):
    def calculate_mismatch_score(row):
        advantages = []
        
        # Primary Metrics (100 baseline)
        primary_metrics = {
            'Penetrator': ('penetrator_score', 'penetrator_score_def'),
            'Pure Score': ('pure_score', 'pure_score_def'),
            'Rim Runner': ('rim_runner_score', 'rim_runner_score_def')
        }
        
        # Calculate primary metric advantages
        for metric_name, (off_col, def_col) in primary_metrics.items():
            if row[off_col] > 100 and row[def_col] > 100:
                advantages.append({
                    'type': metric_name,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'advantage': (row[off_col] - 100) * ((row[def_col] - 100)/100),
                    'baseline': 100
                })

        # Play Type Metrics (50 baseline)
        play_types = {
            'Cut': ('OVERALL_SCORE_Cut', 'OVERALL_DEF_SCORE_Cut_def'),
            'Handoff': ('OVERALL_SCORE_Handoff', 'OVERALL_DEF_SCORE_Handoff_def'),
            'Isolation': ('OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def'),
            'Off Screen': ('OVERALL_SCORE_OffScreen', 'OVERALL_DEF_SCORE_OffScreen_def'),
            'P&R Ball Handler': ('OVERALL_SCORE_PRBallHandler', 'OVERALL_DEF_SCORE_PRBallHandler_def'),
            'P&R Roll Man': ('OVERALL_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_PRRollMan_def'),
            'Post Up': ('OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def'),
            'Spot Up': ('OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def'),
            'Transition': ('OVERALL_SCORE_Transition', 'OVERALL_DEF_SCORE_Transition_def')
        }
        
        # Calculate play type advantages using same formula style
        for play_type, (off_col, def_col) in play_types.items():
            if row[off_col] > 50 and row[def_col] > 50:
                advantages.append({
                    'type': play_type,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'advantage': (row[off_col] - 50) * ((row[def_col] - 50)/50),
                    'baseline': 50
                })

        total_advantage = sum(adv['advantage'] for adv in advantages)
        
        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'total_advantage': total_advantage,
            'pts_avg': row['PTS_60G_Mavg']
        }

    results = []
    for _, row in df.iterrows():
        analysis = calculate_mismatch_score(row)
        if analysis['advantages']:  # Only include players with advantages
            results.append(analysis)
    
    # Sort by total advantage
    results.sort(key=lambda x: x['total_advantage'], reverse=True)
    return results

# Analyze matchups
matchups = analyze_matchups_with_advantages(pts_scores_df)

# Convert to DataFrame for easier analysis
def create_detailed_df(matchups):
    rows = []
    for m in matchups:
        for adv in m['advantages']:
            rows.append({
                'player': m['player'],
                'team': m['team'],
                'opponent': m['opponent'],
                'pts_avg': m['pts_avg'],
                'matchup_type': adv['type'],
                'offensive_rating': adv['offensive_rating'],
                'defensive_rating': adv['defensive_rating'],
                'advantage': adv['advantage'],
                'baseline': adv['baseline']
            })
    return pd.DataFrame(rows)

detailed_df = create_detailed_df(matchups)

print("\nTop Matchup Advantages:")
print("-" * 80)
for i, result in enumerate(matchups[:10], 1):
    print(f"\n{i}. {result['player']} ({result['team']} vs {result['opponent']})")
    print(f"Season Average: {result['pts_avg']:.1f} PPG")
    print(f"Total Advantage Score: {result['total_advantage']:.2f}")
    print("\nIndividual Advantages:")
    
    # Sort advantages by advantage score
    sorted_advantages = sorted(result['advantages'], key=lambda x: x['advantage'], reverse=True)
    
    for adv in sorted_advantages:
        print(f"\n- {adv['type']}:")
        print(f"  Offense: {adv['offensive_rating']:.1f} vs Defense: {adv['defensive_rating']:.1f}")
        print(f"  Advantage Score: {adv['advantage']:.2f}")

# Show top advantages by play type
print("\nTop 5 Individual Matchup Advantages:")
print(detailed_df.nlargest(5, 'advantage')[
    ['player', 'matchup_type', 'offensive_rating', 'defensive_rating', 'advantage']
].to_string())


Top Matchup Advantages:
--------------------------------------------------------------------------------

1. Nikola Jokić (Denver Nuggets vs Brooklyn Nets)
Season Average: 28.8 PPG
Total Advantage Score: 77.84

Individual Advantages:

- Rim Runner:
  Offense: 197.5 vs Defense: 134.9
  Advantage Score: 34.01

- Post Up:
  Offense: 88.1 vs Defense: 83.0
  Advantage Score: 25.15

- Cut:
  Offense: 75.3 vs Defense: 80.4
  Advantage Score: 15.38

- P&R Roll Man:
  Offense: 72.6 vs Defense: 55.7
  Advantage Score: 2.58

- Transition:
  Offense: 55.0 vs Defense: 54.1
  Advantage Score: 0.41

- Spot Up:
  Offense: 53.5 vs Defense: 54.2
  Advantage Score: 0.29

- Pure Score:
  Offense: 102.1 vs Defense: 100.9
  Advantage Score: 0.02

2. Nikola Vučević (Chicago Bulls vs Washington Wizards)
Season Average: 19.7 PPG
Total Advantage Score: 35.63

Individual Advantages:

- P&R Roll Man:
  Offense: 80.5 vs Defense: 79.1
  Advantage Score: 17.75

- Rim Runner:
  Offense: 175.5 vs Defense: 118.7
  Adv

In [2554]:
detailed_df

,player,team,opponent,pts_avg,matchup_type,offensive_rating,defensive_rating,advantage,baseline
0,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Pure Score,102.12,100.92,0.019504,100
1,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Rim Runner,197.48,134.89,34.010772,100
2,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Cut,75.30,80.40,15.382400,50
3,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,P&R Roll Man,72.60,55.70,2.576400,50
4,Nikola Jokić,Denver Nuggets,Brooklyn Nets,28.75,Post Up,88.10,83.00,25.146000,50
...,...,...,...,...,...,...,...,...,...
206,Gary Trent Jr.,Milwaukee Bucks,Orlando Magic,13.00,Handoff,50.90,56.50,0.117000,50
207,Patrick Williams,Chicago Bulls,Washington Wizards,10.62,Spot Up,64.40,50.30,0.086400,50
208,Tristan da Silva,Orlando Magic,Milwaukee Bucks,8.83,Spot Up,53.10,50.60,0.037200,50
209,CJ McCollum,New Orleans Pelicans,Philadelphia 76ers,21.12,Transition,58.40,50.10,0.016800,50


In [2555]:
def analyze_matchups_with_weighted_grade(df):
    def calculate_weighted_score(row):
        advantages = []
        player_strengths = {}
        total_weighted_advantage = 0
        
        # Calculate player strength scores (how good they are at each metric)
        def get_strength_weight(rating, baseline):
            if rating <= baseline:
                return 0
            excess = rating - baseline
            # Exponential weight increase for higher ratings
            return (excess / baseline) ** 1.5

        # Primary Metrics (100 baseline)
        primary_metrics = {
            'Penetrator': ('penetrator_score', 'penetrator_score_def'),
            'Pure Score': ('pure_score', 'pure_score_def'),
            'Rim Runner': ('rim_runner_score', 'rim_runner_score_def')
        }
        
        # Calculate primary metrics advantages and weights
        for metric_name, (off_col, def_col) in primary_metrics.items():
            if row[off_col] > 100 and row[def_col] > 100:
                advantage = (row[off_col] - 100) * ((row[def_col] - 100)/100)
                weight = get_strength_weight(row[off_col], 100)
                weighted_advantage = advantage * weight
                
                advantages.append({
                    'type': metric_name,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'raw_advantage': advantage,
                    'weight': weight,
                    'weighted_advantage': weighted_advantage,
                    'baseline': 100
                })
                total_weighted_advantage += weighted_advantage
                player_strengths[metric_name] = weight

        # Play Type Metrics (50 baseline)
        play_types = {
            'Cut': ('OVERALL_SCORE_Cut', 'OVERALL_DEF_SCORE_Cut_def'),
            'Handoff': ('OVERALL_SCORE_Handoff', 'OVERALL_DEF_SCORE_Handoff_def'),
            'Isolation': ('OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def'),
            'Off Screen': ('OVERALL_SCORE_OffScreen', 'OVERALL_DEF_SCORE_OffScreen_def'),
            'P&R Ball Handler': ('OVERALL_SCORE_PRBallHandler', 'OVERALL_DEF_SCORE_PRBallHandler_def'),
            'P&R Roll Man': ('OVERALL_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_PRRollMan_def'),
            'Post Up': ('OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def'),
            'Spot Up': ('OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def'),
            'Transition': ('OVERALL_SCORE_Transition', 'OVERALL_DEF_SCORE_Transition_def')
        }
        
        # Calculate play type advantages and weights
        for play_type, (off_col, def_col) in play_types.items():
            if row[off_col] > 50 and row[def_col] > 50:
                advantage = (row[off_col] - 50) * ((row[def_col] - 50)/50)
                weight = get_strength_weight(row[off_col], 50)
                weighted_advantage = advantage * weight
                
                advantages.append({
                    'type': play_type,
                    'offensive_rating': row[off_col],
                    'defensive_rating': row[def_col],
                    'raw_advantage': advantage,
                    'weight': weight,
                    'weighted_advantage': weighted_advantage,
                    'baseline': 50
                })
                total_weighted_advantage += weighted_advantage
                player_strengths[play_type] = weight

        # Calculate overall grade (0-100 scale)
        max_possible_advantage = 100  # Theoretical maximum advantage
        overall_grade = min(100, (total_weighted_advantage / max_possible_advantage) * 100)
        
        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'player_strengths': player_strengths,
            'total_weighted_advantage': total_weighted_advantage,
            'overall_grade': overall_grade,
            'pts_avg': row['PTS_60G_Mavg']
        }

    results = []
    for _, row in df.iterrows():
        analysis = calculate_weighted_score(row)
        if analysis['advantages']:
            results.append(analysis)
    
    # Sort by overall grade
    results.sort(key=lambda x: x['overall_grade'], reverse=True)
    return results

# Analyze matchups
matchups = analyze_matchups_with_weighted_grade(pts_scores_df)

print("\nMatchup Grades (Weighted by Player Strengths):")
print("-" * 80)
for i, result in enumerate(matchups[:10], 1):
    print(f"\n{i}. {result['player']} ({result['team']} vs {result['opponent']})")
    print(f"Overall Grade: {result['overall_grade']:.1f}/100")
    print(f"Season Average: {result['pts_avg']:.1f} PPG")
    print("\nKey Advantages (Weighted by Player Strength):")
    
    # Sort advantages by weighted advantage
    sorted_advantages = sorted(result['advantages'], key=lambda x: x['weighted_advantage'], reverse=True)
    
    for adv in sorted_advantages[:3]:  # Show top 3 advantages
        print(f"\n- {adv['type']}:")
        print(f"  Offense: {adv['offensive_rating']:.1f} vs Defense: {adv['defensive_rating']:.1f}")
        print(f"  Raw Advantage: {adv['raw_advantage']:.2f}")
        print(f"  Player Strength Weight: {adv['weight']:.2f}x")
        print(f"  Weighted Advantage: {adv['weighted_advantage']:.2f}")


Matchup Grades (Weighted by Player Strengths):
--------------------------------------------------------------------------------

1. Nikola Jokić (Denver Nuggets vs Brooklyn Nets)
Overall Grade: 55.8/100
Season Average: 28.8 PPG

Key Advantages (Weighted by Player Strength):

- Rim Runner:
  Offense: 197.5 vs Defense: 134.9
  Raw Advantage: 34.01
  Player Strength Weight: 0.96x
  Weighted Advantage: 32.73

- Post Up:
  Offense: 88.1 vs Defense: 83.0
  Raw Advantage: 25.15
  Player Strength Weight: 0.67x
  Weighted Advantage: 16.73

- Cut:
  Offense: 75.3 vs Defense: 80.4
  Raw Advantage: 15.38
  Player Strength Weight: 0.36x
  Weighted Advantage: 5.54

2. Nikola Vučević (Chicago Bulls vs Washington Wizards)
Overall Grade: 18.3/100
Season Average: 19.7 PPG

Key Advantages (Weighted by Player Strength):

- Rim Runner:
  Offense: 175.5 vs Defense: 118.7
  Raw Advantage: 14.12
  Player Strength Weight: 0.66x
  Weighted Advantage: 9.27

- P&R Roll Man:
  Offense: 80.5 vs Defense: 79.1
  Raw

In [2556]:
def analyze_matchups_with_weighted_grade(df):
    def calculate_weighted_score(row):
        advantages = []
        
        # Primary Metrics (100 baseline)
        primary_metrics = {
            'Penetrator': ('penetrator_score', 'penetrator_score_def'),
            'Pure Score': ('pure_score', 'pure_score_def'),
            'Rim Runner': ('rim_runner_score', 'rim_runner_score_def')
        }
        
        # Calculate primary metrics advantages - include all matchups
        for metric_name, (off_col, def_col) in primary_metrics.items():
            # Calculate advantage even if below baseline
            advantage = (row[off_col] - 100) * ((row[def_col] - 100)/100)
            weight = abs((row[off_col] - 100) / 100) ** 1.5  # Use absolute difference for weight
            weighted_advantage = advantage * weight
            
            advantages.append({
                'type': metric_name,
                'offensive_rating': row[off_col],
                'defensive_rating': row[def_col],
                'raw_advantage': advantage,
                'weight': weight,
                'weighted_advantage': weighted_advantage,
                'baseline': 100
            })

        # Play Type Metrics (50 baseline)
        play_types = {
            'Cut': ('OVERALL_SCORE_Cut', 'OVERALL_DEF_SCORE_Cut_def'),
            'Handoff': ('OVERALL_SCORE_Handoff', 'OVERALL_DEF_SCORE_Handoff_def'),
            'Isolation': ('OVERALL_SCORE_Isolation', 'OVERALL_DEF_SCORE_Isolation_def'),
            'Off Screen': ('OVERALL_SCORE_OffScreen', 'OVERALL_DEF_SCORE_OffScreen_def'),
            'P&R Ball Handler': ('OVERALL_SCORE_PRBallHandler', 'OVERALL_DEF_SCORE_PRBallHandler_def'),
            'P&R Roll Man': ('OVERALL_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_PRRollMan_def'),
            'Post Up': ('OVERALL_SCORE_Postup', 'OVERALL_DEF_SCORE_Postup_def'),
            'Spot Up': ('OVERALL_SCORE_Spotup', 'OVERALL_DEF_SCORE_Spotup_def'),
            'Transition': ('OVERALL_SCORE_Transition', 'OVERALL_DEF_SCORE_Transition_def')
        }
        
        # Calculate play type advantages - include all matchups
        for play_type, (off_col, def_col) in play_types.items():
            advantage = (row[off_col] - 50) * ((row[def_col] - 50)/50)
            weight = abs((row[off_col] - 50) / 50) ** 1.5  # Use absolute difference for weight
            weighted_advantage = advantage * weight
            
            advantages.append({
                'type': play_type,
                'offensive_rating': row[off_col],
                'defensive_rating': row[def_col],
                'raw_advantage': advantage,
                'weight': weight,
                'weighted_advantage': weighted_advantage,
                'baseline': 50
            })

        # Calculate overall grade (scale to 0-100 range)
        total_weighted_advantage = sum(adv['weighted_advantage'] for adv in advantages)
        max_possible_advantage = 100
        overall_grade = 50 + (total_weighted_advantage / max_possible_advantage) * 50  # Center at 50

        return {
            'player': row['PLAYER_NAME'],
            'team': row['TEAM_NAME'],
            'opponent': row['OPPONENT_NAME'],
            'advantages': advantages,
            'total_weighted_advantage': total_weighted_advantage,
            'overall_grade': min(100, max(0, overall_grade)),  # Clamp between 0 and 100
            'pts_avg': row['PTS_60G_Mavg'],  # Clamp between 0 and 100
            'min_avg': row['MIN_60G_Mavg']
        }

    results = []
    for _, row in df.iterrows():
        analysis = calculate_weighted_score(row)
        results.append(analysis)
    
    return results

# Create DataFrames
def create_detailed_df(matchups):
    rows = []
    for m in matchups:
        for adv in m['advantages']:
            rows.append({
                'player': m['player'],
                'team': m['team'],
                'opponent': m['opponent'],
                'pts_avg': m['pts_avg'],
                'overall_grade': m['overall_grade'],
                'matchup_type': adv['type'],
                'offensive_rating': adv['offensive_rating'],
                'defensive_rating': adv['defensive_rating'],
                'raw_advantage': adv['raw_advantage'],
                'player_strength_weight': adv['weight'],
                'weighted_advantage': adv['weighted_advantage'],
                'baseline': adv['baseline']
            })
    return pd.DataFrame(rows)

def create_summary_df(matchups):
    summary_data = []
    for m in matchups:
        player_data = {
            'player': m['player'],
            'team': m['team'],
            'opponent': m['opponent'],
            'min_avg': m['min_avg'],
            'pts_avg': m['pts_avg'],
            'overall_grade': m['overall_grade']
        }
        
        # Add each matchup type's weighted advantage
        for adv in m['advantages']:
            col_name = f"{adv['type']}_weighted_grade"
            player_data[col_name] = adv['weighted_advantage']
            
        summary_data.append(player_data)
    
    df = pd.DataFrame(summary_data)
    return df.sort_values('overall_grade', ascending=False)

# Create both DataFrames
matchups = analyze_matchups_with_weighted_grade(pts_scores_df)
detailed_df = create_detailed_df(matchups)
summary_df = create_summary_df(matchups)



In [2557]:
summary_df

,player,team,opponent,min_avg,pts_avg,overall_grade,Penetrator_weighted_grade,Pure Score_weighted_grade,Rim Runner_weighted_grade,Cut_weighted_grade,Handoff_weighted_grade,Isolation_weighted_grade,Off Screen_weighted_grade,P&R Ball Handler_weighted_grade,P&R Roll Man_weighted_grade,Post Up_weighted_grade,Spot Up_weighted_grade,Transition_weighted_grade
75,Yves Missi,New Orleans Pelicans,Philadelphia 76ers,27.27,9.19,94.590563,1.561774,1.947701,6.640034,-0.001340,30.700000,-5.200000,10.900000,12.100000,1.553906,15.800000,13.178718,0.000332
11,Cole Anthony,Orlando Magic,Milwaukee Bucks,16.86,8.53,81.255223,0.340486,-0.035222,0.726557,25.900000,0.015807,2.656503,-1.114481,-0.112469,1.604091,32.700000,-0.036969,-0.133857
15,Tristan da Silva,Orlando Magic,Milwaukee Bucks,25.94,8.83,79.290136,-0.256837,0.563034,0.000064,0.963023,2.406137,23.800000,-2.387699,-0.300090,1.166562,32.700000,0.000574,-0.074498
1,Myles Turner,Indiana Pacers,Golden State Warriors,29.12,15.80,79.256192,3.767617,0.032761,-2.818764,-0.123080,2.421477,0.396964,27.000000,30.000000,-2.810749,-0.010315,0.003446,0.653028
8,Kentavious Caldwell-Pope,Orlando Magic,Milwaukee Bucks,31.19,9.65,79.216347,-0.808367,0.308704,0.126318,0.793284,1.158153,23.800000,-0.087913,-0.041802,0.498932,32.700000,-0.000169,-0.014447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,Zion Williamson,New Orleans Pelicans,Philadelphia 76ers,31.45,22.67,33.552393,-5.520456,-0.087099,0.841770,-0.027624,-0.034356,1.997396,0.054373,-0.001886,-30.000000,-0.128688,0.011392,-0.000036
99,Carlton Carrington,Washington Wizards,Chicago Bulls,30.22,9.26,32.488182,-0.065970,-0.045751,-7.957917,-1.669429,-0.050810,-4.912318,-9.600000,0.012251,1.900000,-12.300000,0.066308,-0.400000
60,Trayce Jackson-Davis,Golden State Warriors,Indiana Pacers,20.91,9.38,27.928556,-0.885462,-10.749127,2.128290,-0.446618,-6.100000,-12.500000,-10.200000,-3.900000,0.006336,1.401284,-2.900000,0.002409
42,Talen Horton-Tucker,Chicago Bulls,Washington Wizards,15.43,8.62,27.023218,-0.772231,0.023591,-3.856670,-13.800000,0.031925,-0.723648,2.074454,0.000480,-29.100000,0.209490,-0.018204,-0.022752
